In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
# PIPELINE_RUN_ID: blank = generated by start_run(); a Fabric pipeline can pass one in.
PIPELINE_RUN_ID = ''

In [ ]:
%run ./NB_Utils_Common

In [ ]:
%run ./NB_Config_PROC

In [ ]:
%run ./NB_Utils_Gold

In [ ]:
%run ./NB_Utils_Gold_DQ

In [ ]:
# ── Run setup: run ID, Spark settings, table locations ─────────────────────────
start_run('PROC_Gold', 'GOLD')

In [ ]:
# ── Gold settings ─────────────────────────────────────────────────────────────
# REDUNDANT_COLS: natural keys and description columns already covered by SKs (always dropped).
REDUNDANT_COLS = [
    'Company_Code_Name', 'Purch_Organization_Name', 'Purchasing_Group_Name', 'Plant_Name',
    'Material', 'Plant', 'Company_Code', 'Supplier', 'Purch_Organization', 'Purchasing_Group',
]

# GOLD_DIMS: MD Gold dimensions the referential-integrity check compares the facts' SKs with.
GOLD_DIMS = {
    'Vendor_SK':           ('md_gold', 'Dim_Vendor'),
    'Material_SK':         ('md_gold', 'Dim_Material'),
    'Plant_SK':            ('md_gold', 'Dim_Plant'),
    'Company_Code_SK':     ('md_gold', 'Dim_Company_Code'),
    'Purch_Org_SK':        ('md_gold', 'Dim_Purch_Org'),
    'Purchasing_Group_SK': ('md_gold', 'Dim_Purchasing_Group'),
    'PIR_Purch_Org_SK':    ('md_gold', 'Dim_PIR'),
}

# GOLD_DQ: overrides for run_gold_dq (none: each table's driving Silver table owns its SK).
GOLD_DQ = {}

In [ ]:
# ── Load active rows from Silver ──────────────────────────────────────────────

ekko = silver('EKKO')
ekpo = silver('EKPO')   # carries Price_Per_Unit, Total_Stock, Safety_Stock from Silver enrichment
eket = silver('EKET')
ekbe = silver('EKBE')
ekes = silver('EKES')   # vendor confirmations
ekkn = silver('EKKN')   # account assignment
ekpa = silver('EKPA')   # partner functions
ekab = silver('EKAB')   # contract release documentation

print('Silver tables loaded')

In [ ]:
print('\nBuilding Fact_PO_LineItem...')
# NOTE 25 Oct 2026: DUPLICATE of Fact_PO_Delivery_Status (same EKPO + EKKO item rows). Its two extras, the EKKN account assignment
# and the MBEW moving price, are now on Fact_PO_Delivery_Status. Kept only until the reports / semantic model are repointed;
# then delete this cell and the name in NB_Cleanup_PROC / the EKKO and EKPO gold_name labels.

# Rename header-level columns that also exist in EKPO before computing header_override.
# Without this, Incoterms/Incoterms_2/Deletion_Indicator from EKPO are silently dropped.
ekko_dedup = (ekko
    .withColumnRenamed('Deletion_Indicator', 'Header_Deletion_Indicator')
    .withColumnRenamed('Incoterms',          'Header_Incoterms')
    .withColumnRenamed('Incoterms_2',        'Header_Incoterms_2')
)
header_override = [c for c in ekko_dedup.columns
                   if c not in ('PO_SK', 'Purchasing_Document', '_CREATED_AT', '_UPDATED_AT')
                   and c in ekpo.columns]
ekpo_clean = ekpo.drop(*header_override)
ekko_slim  = ekko_dedup.select([c for c in ekko_dedup.columns if c not in ('_CREATED_AT', '_UPDATED_AT')])

fact_li = ekpo_clean.join(ekko_slim, on='Purchasing_Document', how='left')

# Account assignment (EKKN): order and G/L account of the first assignment per item;
# Acct_Assignment_Count > 1 means the item is split across several assignments.
_w_kn = Window.partitionBy('Purchasing_Document', 'Item').orderBy('Acct_Assignment_No')
acct = (ekkn.withColumn('_rn', F.row_number().over(_w_kn))
            .withColumn('Acct_Assignment_Count', F.count('*').over(Window.partitionBy('Purchasing_Document', 'Item')))
            .filter(F.col('_rn') == 1)
            .select('Purchasing_Document', 'Item', 'Order', 'GL_Account', 'Acct_Assignment_Count'))
fact_li = fact_li.join(acct, ['Purchasing_Document', 'Item'], 'left')

sk_cols   = [c for c in ('PO_Item_SK','PO_SK','Vendor_SK','Material_SK',
                          'Plant_SK','Company_Code_SK','Purch_Org_SK','Purchasing_Group_SK',
                          'Order_From_Vendor_SK')
             if c in fact_li.columns]
pk_cols   = ['Purchasing_Document', 'Item']
rest_cols = [c for c in fact_li.columns if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in fact_li.columns if c.startswith('_')]
fact_li   = fact_li.select(sk_cols + pk_cols + rest_cols + meta_cols)

# EA audit reports (phase 1): the material's moving price and price unit at the PO plant (MD Silver MBEW, unique on
# material + plant). PO_SK / PO_Item_SK are the keys SM_AUDIT relates on (the Audit GL fact gets them from PROC Silver).
_mbew = (read_table('md_silver', 'MBEW', active=True)
         .select(_NZ(F.col('Material')).alias('_mb_mat'), _NZ(F.col('Plant')).alias('_mb_plant'),
                 F.col('Moving_Avg_Price').alias('Material_Moving_Price'), F.col('Price_Unit').alias('Material_Price_Unit'))
         .dropDuplicates(['_mb_mat', '_mb_plant']))
fact_li = (fact_li
    .join(_mbew, (_NZ(F.col('Material')) == F.col('_mb_mat')) & (_NZ(F.col('Plant')) == F.col('_mb_plant')), 'left')
    .drop('_mb_mat', '_mb_plant'))

fact_li = drop_redundant(fact_li)
n = write_gold(fact_li, 'Fact_PO_LineItem', 'PO_Item_SK')

In [ ]:
print('\nBuilding Fact_PO_Schedule...')

# PO_SK and PO_Item_SK (relate to Dim_PO_Document) are resolved in Silver (EKET sk_joins)
fact_sch = eket

sk_cols   = [c for c in ('PO_Schedule_SK', 'PO_Item_SK', 'PO_SK') if c in fact_sch.columns]
pk_cols   = ['Purchasing_Document', 'Item', 'Schedule_Line']
rest_cols = [c for c in fact_sch.columns if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in fact_sch.columns if c.startswith('_')]
fact_sch  = fact_sch.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_sch, 'Fact_PO_Schedule', 'PO_Schedule_SK')

In [ ]:
print('\nBuilding Fact_PO_Confirmation...')

# One row per vendor confirmation (EKES). Which confirmation is the "Order Due Date" is report logic.
# PO_SK / PO_Item_SK resolved in Silver (EKES sk_joins)
fact_conf = ekes

sk_cols   = [c for c in ('PO_Confirmation_SK', 'PO_Item_SK', 'PO_SK') if c in fact_conf.columns]
pk_cols   = ['Purchasing_Document', 'Item', 'Confirmation_Seq']
rest_cols = [c for c in fact_conf.columns if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in fact_conf.columns if c.startswith('_')]
fact_conf = fact_conf.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_conf, 'Fact_PO_Confirmation', 'PO_Confirmation_SK')

In [ ]:
print('\nBuilding Fact_PO_Contract_Release...')

# One row per release order against a contract item (EKAB); PO SKs are those of the release order.
# PO_SK / PO_Item_SK of the release order resolved in Silver (EKAB sk_joins)
fact_rel = ekab

sk_cols   = [c for c in ('Contract_Release_SK', 'PO_Item_SK', 'PO_SK') if c in fact_rel.columns]
pk_cols   = ['Contract', 'Contract_Item', 'Purchasing_Document', 'Item']
rest_cols = [c for c in fact_rel.columns if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in fact_rel.columns if c.startswith('_')]
fact_rel  = fact_rel.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_rel, 'Fact_PO_Contract_Release', 'Contract_Release_SK')

In [ ]:
print('\nBuilding Fact_PO_Receipts...')


# PO_SK / PO_Item_SK (relate to Dim_PO_Document) resolved in Silver (EKBE sk_joins)
fact_rcpt = ekbe

sk_cols   = [c for c in ('PO_History_SK', 'PO_Item_SK', 'PO_SK', 'Material_SK', 'Plant_SK')
             if c in fact_rcpt.columns]
pk_cols   = ['Purchasing_Document', 'Item', 'Seq_No_Of_Account',
             'Transevent_Type', 'Material_Doc_Year', 'Material_Document', 'Material_DocItem']
rest_cols = [c for c in fact_rcpt.columns if c not in sk_cols + pk_cols and not c.startswith('_')]
meta_cols = [c for c in fact_rcpt.columns if c.startswith('_')]
fact_rcpt = fact_rcpt.select(sk_cols + pk_cols + rest_cols + meta_cols)

fact_rcpt = drop_redundant(fact_rcpt)
n = write_gold(fact_rcpt, 'Fact_PO_Receipts', 'PO_History_SK')

In [ ]:
print('\nBuilding Fact_PO_Delivery_Status...')
# The PO item fact: EKPO + EKKO header + EKET totals + status + EKKN account assignment + MBEW price (one row per PO item).

ekpo_open = silver('EKPO')

# Join EKKO header; rename shared cols to distinguish header-level from item-level values
ekko_for_join = (ekko
    .withColumnRenamed('Deletion_Indicator', 'Header_Deletion_Indicator')
    .withColumnRenamed('Incoterms',          'Header_Incoterms')
    .withColumnRenamed('Incoterms_2',        'Header_Incoterms_2')
)
header_override = [c for c in ekko_for_join.columns
                   if c not in ('PO_SK', 'Purchasing_Document', '_CREATED_AT', '_UPDATED_AT')
                   and c in ekpo_open.columns]
ekpo_open = ekpo_open.drop(*header_override)
base = ekpo_open.join(
    ekko_for_join.select([c for c in ekko_for_join.columns if c not in ('_CREATED_AT', '_UPDATED_AT')]),
    on='Purchasing_Document', how='left')

# Aggregate EKET: Delivered and Scheduled quantity per PO line
# ▶ 19 Nov 2026 (Option A, agreed with the user): a zero-quantity schedule line (EKET MENGE 0) carries a copy of the receipt booked on
#   another line of the same item (13219607 item 2: both lines WEMNG 1,200, EKBE 1,200), so adding WEMNG over all lines counted the
#   receipt twice and closed items that are only partly received. Delivered now adds WEMNG only over lines with a quantity; when a
#   zero line holds more than that sum (a receipt found on no other line), its WEMNG is kept so no receipt is lost. An item whose
#   lines are all zero keeps the sum over all its lines (nothing is doubled there). The spec rule (SAP table list v14:
#   Quantity_Open = ordered - Σ EKET WEMNG) is kept. Delivered, Quantity_Open and Execution_Status change here and flow to
#   Fact_PO_Header and Fact_PO_LineItem_Schedule; no column is added or removed.
_has_qty = F.coalesce(F.col('Scheduled_Quantity'), F.lit(0)) != 0
eket_agg = (eket.groupBy('Purchasing_Document', 'Item')
                .agg(F.sum(F.when(_has_qty, F.col('Quantity_Delivered'))).alias('_dlv_qty_lines'),
                     F.max(F.when(~_has_qty, F.col('Quantity_Delivered'))).alias('_dlv_zero_max'),
                     F.sum('Quantity_Delivered').alias('_dlv_all_lines'),
                     F.max(_has_qty.cast('int')).alias('_any_qty_line'),
                     F.sum('Scheduled_Quantity').alias('Scheduled_Qty'))
                .withColumn('Delivered', F.when(F.col('_any_qty_line') == 1, F.greatest('_dlv_qty_lines', '_dlv_zero_max'))
                                          .otherwise(F.col('_dlv_all_lines'))))
# Warn-only: items whose receipt total drops because a zero-quantity line repeated it (counted, never forced)
_n_zero_dup = eket_agg.filter(F.coalesce(F.col('_dlv_all_lines'), F.lit(0)) != F.coalesce(F.col('Delivered'), F.lit(0))).count()
print(f'  Items whose receipt was repeated on a zero-quantity line (now counted once): {_n_zero_dup:,}')
dq_record('GOLD', 'Fact_PO_Delivery_Status', 'zero_line_receipt', 'WARN' if _n_zero_dup else 'PASS',
          f'{_n_zero_dup:,} items had their receipt repeated on a zero-quantity schedule line; Delivered counts it once (Option A)',
          observed=_n_zero_dup)
eket_agg = eket_agg.drop('_dlv_qty_lines', '_dlv_zero_max', '_dlv_all_lines', '_any_qty_line')
base = base.join(eket_agg, on=['Purchasing_Document', 'Item'], how='left')

# Account assignment (EKKN) and the material's moving price (MD Silver MBEW): moved here from Fact_PO_LineItem on 25 Oct 2026,
# which held nothing else of its own (see the note on that cell). Order and G/L account of the first assignment per item;
# Acct_Assignment_Count > 1 means the item is split across several assignments. Dim_PO_Item reads `acct` too.
_w_kn = Window.partitionBy('Purchasing_Document', 'Item').orderBy('Acct_Assignment_No')
acct = (ekkn.withColumn('_rn', F.row_number().over(_w_kn))
            .withColumn('Acct_Assignment_Count', F.count('*').over(Window.partitionBy('Purchasing_Document', 'Item')))
            .filter(F.col('_rn') == 1)
            .select('Purchasing_Document', 'Item', 'Order', 'GL_Account', 'Acct_Assignment_Count'))
base = base.join(acct, ['Purchasing_Document', 'Item'], 'left')

# The moving price and price unit at the PO plant (MBEW is unique on material + plant)
_mbew = (read_table('md_silver', 'MBEW', active=True)
         .select(_NZ(F.col('Material')).alias('_mb_mat'), _NZ(F.col('Plant')).alias('_mb_plant'),
                 F.col('Moving_Avg_Price').alias('Material_Moving_Price'), F.col('Price_Unit').alias('Material_Price_Unit'))
         .dropDuplicates(['_mb_mat', '_mb_plant']))
base = (base
    .join(_mbew, (_NZ(F.col('Material')) == F.col('_mb_mat')) & (_NZ(F.col('Plant')) == F.col('_mb_plant')), 'left')
    .drop('_mb_mat', '_mb_plant'))

# Derive business status columns
#   Execution_Status: header/line cancellation → closed → partial → open
#   Price_per_Unit:   project from EKPO Silver (already computed there — no recompute at Gold)
#   Quantity_Open:    ordered (EKPO MENGE) minus delivered (Σ EKET WEMNG), as in the SAP table list v14
#   Below_Safety_Stock: flag when on-hand stock falls below Safety_Stock threshold
base = (base
    .withColumn('Execution_Status',
        F.when((F.col('Header_Deletion_Indicator') == 'L') | (F.col('Header_Deletion_Indicator') == 'C'), 'Cancelled_Header')
         .when(F.col('Deletion_Indicator')         == 'L', 'Cancelled_Line')
         .when(F.col('Delivery_Completed')          == 'X', 'Closed')
         .when((F.col('Delivered') > 0) & (F.col('Delivered') < F.col('Scheduled_Qty')), 'Partially Open')
         .when(F.col('Delivered') >= F.col('Scheduled_Qty'), 'Closed')
         .otherwise('Open'))
    .withColumn('Quantity_Open',
        F.col('Order_Quantity') - F.coalesce(F.col('Delivered'), F.lit(0)))
    .withColumn('Below_Safety_Stock',
        F.when(F.col('Total_Stock') < F.col('Safety_Stock'), 'Yes').otherwise('No'))
)

# Column ordering and write
sk_cols   = [c for c in ('PO_Item_SK','PO_SK','Vendor_SK','Material_SK',
                          'Plant_SK','Company_Code_SK','Purch_Org_SK','Purchasing_Group_SK','Purchase_Requisition_SK','PO_Stock_SK',)
             if c in base.columns]
pk_cols   = ['Purchasing_Document', 'Item']
calc_cols = ['Execution_Status','Header_Deletion_Indicator','Deletion_Indicator',
             'Header_Incoterms','Incoterms','Header_Incoterms_2','Incoterms_2',
             'Quantity_Open','Price_Per_Unit','Delivered','Scheduled_Qty',
             'Total_Stock','Safety_Stock','Below_Safety_Stock']
rest_cols = [c for c in base.columns if c not in sk_cols + pk_cols + calc_cols and not c.startswith('_')]
meta_cols = [c for c in base.columns if c.startswith('_')]
base = base.select(sk_cols + pk_cols + calc_cols + rest_cols + meta_cols)

base = drop_redundant(base)
n = write_gold(base, 'Fact_PO_Delivery_Status', 'PO_Item_SK')

In [ ]:
# ── Dim_PO_Document (PO+Item grain — hub, keys + SKs only) ───────────────────
# EKPO + EKKO joined to collect all SKs at item grain. No attributes passed through
# — descriptive content lives in Dim_Vendor, Dim_Material, Dim_Plant, etc.
# PO_Item_SK is the PK; the PROC facts relate to this hub on PO_Item_SK.

print('\nBuilding Dim_PO_Document...')

ekpo_sv = silver('EKPO')

ekko_sv = silver('EKKO')

ekko_keys = ekko_sv.select('Purchasing_Document', 'PO_SK',
                             'Vendor_SK', 'Company_Code_SK', 'Purch_Org_SK', 'Purchasing_Group_SK')

dim_po_doc = (ekpo_sv
    .select('Purchasing_Document', 'Item', 'PO_Item_SK', 'Material_SK', 'Plant_SK')
    .join(ekko_keys, 'Purchasing_Document', 'left'))

sk_cols = [c for c in ('PO_Item_SK', 'PO_SK', 'Vendor_SK', 'Material_SK',
                        'Plant_SK', 'Company_Code_SK', 'Purch_Org_SK', 'Purchasing_Group_SK')
           if c in dim_po_doc.columns]
dim_po_doc = dim_po_doc.select(sk_cols + ['Purchasing_Document', 'Item'])

n = write_gold(dim_po_doc, 'Dim_PO_Document', 'PO_Item_SK')

In [ ]:
# ── Dim_PO_Header, Dim_PO_Item, Fact_PO_Header (EA audit reports, phase 1) ─────────────────────────────
# Next to Dim_PO_Document, which is unchanged. Keyed on PO_SK / PO_Item_SK (PROC Silver); the Audit GL fact gets the
# same SKs from PROC Silver (SK exception to the domain rule), so SM_AUDIT relates facts of both domains to them.
#   Dim_PO_Header : one row per PO (EKKO) — type, category, dates, payment terms, release data, order-from vendor
#   Dim_PO_Item   : one row per PO item (EKPO + first EKKN account assignment, as on Fact_PO_LineItem)
#   Fact_PO_Header: one row per PO (report 09 grain): keys and Execution_Status rolled up from the items; header
#                   amounts and the other roll-ups come in phase 2
print('\nBuilding Dim_PO_Header...')
_zterm = (read_table('md_silver', 'T052U', active=True)
          .select(F.trim(F.col('ZTERM')).alias('_zt'), F.col('TEXT1').alias('Payment_Terms_Desc')).dropDuplicates(['_zt']))
dim_po_hdr = (ekko
    .join(_zterm, F.trim(F.col('Payment_Terms')) == F.col('_zt'), 'left')
    .select('PO_SK', 'Purchasing_Document', 'Company_Code',
            'Purchasing_Doc_Type', 'Purchasing_Doc_Type_Desc', 'Purch_Doc_Category',
            'Created_On', 'Created_By', 'Document_Date', 'Payment_Terms', 'Payment_Terms_Desc',
            'Currency', 'Supplying_Plant',
            'Release_Group', 'Release_Group_Desc', 'Release_Strategy', 'Release_Strategy_Desc',
            'Release_Indicator', 'Release_Indicator_Desc', 'Release_Status', 'Subject_To_Release',
            F.col('Deletion_Indicator').alias('Header_Deletion_Indicator'), 'Order_From_Vendor'))
n = write_gold(dim_po_hdr, 'Dim_PO_Header', 'PO_SK')

print('\nBuilding Dim_PO_Item...')
# NOTE: EKPO no longer has Item_Of_Requisition (it lives on EBAN / EBKN). Drop it from the select list to avoid unresolved-column errors.
dim_po_item = (ekpo
    .join(acct.select('Purchasing_Document', 'Item', 'Order', 'GL_Account'), ['Purchasing_Document', 'Item'], 'left')
    .select('PO_Item_SK', 'Purchasing_Document', 'Item',
            F.concat_ws('/', 'Purchasing_Document', F.col('Item').cast('int').cast('string')).alias('PO_Item_ID'),
            'Short_Text', 'Item_Category', 'Item_Category_Ext', 'Item_Category_Desc', 'Acct_Assignment_Cat',
            'Order_Unit', 'Price_Unit', 'Qty_Conversion_Num', 'Purchase_Requisition',
            # 'Item_Of_Requisition' was removed from EKPO; keep the field at the requisition grain (EBAN / EBKN-based facts/dims)
            'Requisitioner',
            'Goods_Receipt_Ind', 'Invoice_Receipt_Ind', 'GR_Based_Inv_Verif', 'Confirmation_Control',
            'Order_Acknowledgment', 'Subcontracting_Vendor', 'Deletion_Indicator', 'Delivery_Completed',
            'Returns_Item', 'Order', 'GL_Account', 'Req_Tracking_Number'))
            # 'Address' was removed from EKPO (moved to EBAN / EBKN). Drop from this dim to avoid unresolved-column errors.
n = write_gold(dim_po_item, 'Dim_PO_Item', 'PO_Item_SK')

print('\nBuilding Fact_PO_Header...')
# Execution_Status per PO (report 09): rolled up from the item status of Fact_PO_Delivery_Status (written above).
# Header cancelled → Cancelled. Cancelled lines are left out; all remaining lines Closed → Closed; nothing received on
# any of them (all Open) → Open; otherwise Partially Open. A PO whose lines are all cancelled → Cancelled.
_live = ~F.col('Execution_Status').isin('Cancelled_Header', 'Cancelled_Line')
po_execution = (gold('Fact_PO_Delivery_Status')
    .groupBy('PO_SK')
    .agg(F.max(F.col('Execution_Status') == 'Cancelled_Header').alias('_hdr_cancelled'),
         F.sum(F.when(_live, 1).otherwise(0)).alias('_lines'),
         F.sum(F.when(_live & (F.col('Execution_Status') == 'Closed'), 1).otherwise(0)).alias('_closed'),
         F.sum(F.when(_live & (F.col('Execution_Status') == 'Open'), 1).otherwise(0)).alias('_open'))
    .select('PO_SK',
            F.when(F.col('_hdr_cancelled') | (F.col('_lines') == 0), 'Cancelled')
             .when(F.col('_closed') == F.col('_lines'), 'Closed')
             .when(F.col('_open') == F.col('_lines'), 'Open')
             .otherwise('Partially Open').alias('Execution_Status')))
fact_po_hdr = (ekko.select('PO_SK', 'Purchasing_Document',
                           'Vendor_SK', 'Company_Code_SK', 'Purch_Org_SK', 'Purchasing_Group_SK',
                           'Document_Date', 'Created_On', 'Created_By')
               .join(po_execution, 'PO_SK', 'left'))
n = write_gold(fact_po_hdr, 'Fact_PO_Header', 'PO_SK')

In [ ]:
# ── Dim_PO_Bridge — scoped Cartesian product per PO+Item across 3 facts ───────



print('\nBuilding Dim_PO_Bridge...')

eket_keys = eket.select('Purchasing_Document', 'Item', 'PO_Schedule_SK')
ekbe_keys = ekbe.select('Purchasing_Document', 'Item', 'PO_History_SK')
ekpo_keys = ekpo.select('Purchasing_Document', 'Item', 'PO_Item_SK')
ekko_keys = ekko.select('Purchasing_Document', 'PO_SK')

bridge = (eket_keys
    .join(ekbe_keys, ['Purchasing_Document', 'Item'], 'left')
    .join(ekpo_keys, ['Purchasing_Document', 'Item'], 'left')
    .join(ekko_keys, 'Purchasing_Document', 'left')
    .select('PO_SK', 'Purchasing_Document', 'Item',
            'PO_Schedule_SK', 'PO_History_SK', 'PO_Item_SK'))

n = write_gold(bridge, 'Dim_PO_Bridge', mode='overwrite')

### NEW · Order_Status_Purchase (20 Oct 2026, rebuilt as flags 21 Oct 2026)
The cells below build the schedule-line status for `Fact_PO_LineItem_Schedule` in three layers of named yes/no flags (as OTIF does):
1. **Fact flags** (`Is_Open`, `Is_Closed_Line`, `Has_GR_Date`, `Has_Shipping_Conf` ...) for each segment: an EKET line (its unconfirmed remainder) or a confirmed piece (an open EKES confirmation allocated to a line);
2. **Rule flags**, one per rule (`Is_Cancelled` ... `Is_To_Be_Confirmed`), each a short AND / OR of fact flags;
3. **`Order_Status_Rule`**: the first rule flag that is true, in priority order; the status text comes from `OS_RULES`.

Settings (tolerance, EKES categories, `OS_RULES`) are in `NB_Config_PROC`. Every new cell starts with `▶▶ NEW`; changed lines carry `▶ CHANGED`.

**6 Nov 2026 (Jeremy's feedback on the lineage workbook v2).**
- **Allocation (C12):** confirmations are spread cumulatively over the open schedule lines in date order.
- **Stock transfer (C4):** PSTYP U.
- **Categories (C1):** AB means Confirmed; ZS and GW mean Shipping notification.
- **New rule 0, Return item (C20):** reports exclude these lines.
- **Rule 1** also takes EA early exits 6 and 8 (C24).
- **Rules 2 and 3** also close on the under-delivery tolerance (UNTTO) and on ELIKZ. A line blocked after a GR counts as Closed by GR (C2, C21, C22).
- **Rule 6 retired:** movement 107 is an advance shipping notice and now feeds rule 11.
- **Rule 14** uses EA's committed schedule line (C24, which overrides C19).
Details are in `Procurment/PROC_Order_Status_Lineage_v3.xlsx`.

In [ ]:
# ▶▶ NEW 20-Oct-2026 · Order_Status_Purchase · step 1: required Silver columns ◀◀
# Silver silently skips a configured column that Bronze does not have, so check them here and stop with a clear message.
_REQUIRED = {
    'EKET': (eket, ['Schedule_Line', 'Delivery_Date', 'Scheduled_Quantity', 'Quantity_Delivered', 'Qty_Delivered_STO', 'Qty_Reduced_MRP',
                    'Committed_Quantity', 'Committed_Date']),                        # ▶ 6 Nov 2026: Stock committed, EA logic (C24)
    'EKES': (ekes, ['Confirmation_Seq', 'Confirmation_Category', 'Confirmed_Delivery_Date', 'Confirmation_Quantity',
                    'Confirmation_Qty_Reduced_MRP', 'Inbound_Delivery', 'Confirmation_Deletion']),
    'EKBE': (ekbe, ['Confirmation_Seq', 'Transevent_Type', 'PO_History_Category', 'Movement_Type', 'DebitCredit_Ind',
                    'Posting_Date', 'Signed_Quantity']),
    'EKPO': (ekpo, ['Deletion_Indicator', 'Rejection_Indicator', 'Acknowledgment_Reqd', 'Confirmation_Control', 'Order_Acknowledgment',
                    # ▶ 6 Nov 2026 (Jeremy's feedback): stock transfer (C4), returns (C20), delivery completed (C21), tolerance (C22),
                    # final invoice and final delivery (C24, EA early exits 6 and 8)
                    'Item_Category_Ext', 'Returns_Item', 'Delivery_Completed', 'Underdelivery_Tolerance',
                    'Invoice_Receipt_Ind', 'Goods_Receipt_Ind', 'Final_Invoice', 'Final_Delivery']),
    'EKKO': (ekko, ['Deletion_Indicator', 'Purch_Doc_Category']),   # ▶ 6 Nov 2026: Supplying_Plant no longer decides a stock transfer (C4)
}
_missing = {t: [c for c in cols if c not in df.columns] for t, (df, cols) in _REQUIRED.items()}
_missing = {t: m for t, m in _missing.items() if m}
if _missing:
    raise ValueError('Order_Status_Purchase: Silver columns missing (check the Bronze load, then run Silver with RUN_FORCE_FULL = True): '
                     + '; '.join(f'{t}: {m}' for t, m in _missing.items()))
print('Order_Status_Purchase: required Silver columns present')

In [ ]:
# ▶▶ NEW 20-Oct-2026 · Order_Status_Purchase · step 2: item flags, schedule lines and confirmation allocation ◀◀
# ▶ CHANGED 6 Nov 2026 (Jeremy's feedback on the lineage workbook, v2):
#   - C12: open supplier confirmations (EKES) are allocated cumulatively to the open schedule lines (EKET) of the item, in
#     date order, as EA does: 3 lines and 1.5 lines confirmed gives line 1 confirmed, line 2 half confirmed and half not,
#     line 3 not confirmed. A confirmed piece is a segment of its line; the line keeps its unconfirmed remainder.
#     was: each open confirmation was its own segment and the EKET line was reduced by DABMG.
#   - C4: a stock transfer is item category U (EKPO PSTYP, external code EPSTP). was: a supplying plant on the PO header.
#   - C20 / C21 / C22 / C24: return item, delivery completed, under-delivery tolerance, final invoice and final delivery.

# ── Item flags: one named yes/no per fact about the PO item and its header ─────────────────────────────────
po_header = (ekko.select('Purchasing_Document',
                         F.col('Deletion_Indicator').alias('Header_Deletion'),
                         F.col('Purch_Doc_Category').alias('Header_Category'))
             .dropDuplicates(['Purchasing_Document']))

_is_x  = lambda c: F.trim(F.col(c)).eqNullSafe('X')
_blank = lambda c: F.trim(F.coalesce(F.col(c).cast('string'), F.lit(''))) == ''

item_flags = (ekpo.select('Purchasing_Document', 'Item',
                          F.col('Deletion_Indicator').alias('Item_Deletion'),
                          'Rejection_Indicator', 'Acknowledgment_Reqd', 'Confirmation_Control', 'Order_Acknowledgment',
                          'Item_Category_Ext', 'Returns_Item', 'Delivery_Completed', 'Underdelivery_Tolerance',
                          'Invoice_Receipt_Ind', 'Goods_Receipt_Ind', 'Final_Invoice', 'Final_Delivery')
              .dropDuplicates(['Purchasing_Document', 'Item'])
              .join(po_header, 'Purchasing_Document', 'left')
              .withColumns({
                  'Is_Stock_Transfer':      F.trim(F.col('Item_Category_Ext')).eqNullSafe('U'),                # C4: PSTYP U (Jeremy, 6 Nov)
                  'Is_Return_Item':         _is_x('Returns_Item'),                                               # C20: EKPO RETPO
                  'Is_Header_Deleted':      F.col('Header_Deletion').eqNullSafe('L'),
                  'Is_Item_Deleted':        F.col('Item_Deletion').eqNullSafe('L'),
                  'Is_Item_Rejected':       F.col('Rejection_Indicator').eqNullSafe('X'),
                  'Is_Header_Blocked':      F.col('Header_Deletion').eqNullSafe('S'),
                  'Is_Item_Blocked':        F.col('Item_Deletion').eqNullSafe('S'),                          # question C2
                  'Is_Request_Or_Contract': F.coalesce(F.col('Header_Category').isin('A', 'K'), F.lit(False)),   # question C3
                  'Is_Ack_Required':        F.col('Acknowledgment_Reqd').eqNullSafe('X'),
                  'Has_Conf_Control':       ~_blank('Confirmation_Control'),
                  'Has_Ack_Number':         ~_blank('Order_Acknowledgment'),
                  'Is_Delivery_Completed':  _is_x('Delivery_Completed'),                                         # C21: ELIKZ closes the line
                  # C24, EA early exit 6: final invoice posted on an item that is not GR-relevant (REPOS = X, WEPOS blank, EREKZ = X)
                  'Is_Final_Invoice_No_GR': _is_x('Invoice_Receipt_Ind') & _blank('Goods_Receipt_Ind') & _is_x('Final_Invoice'),
                  'Is_Final_Delivery':      _is_x('Final_Delivery'),                                             # C24, EA early exit 8 (EGLKZ)
                  'Underdelivery_Pct':      F.coalesce(F.col('Underdelivery_Tolerance').cast('double'), F.lit(0.0)),   # C22: UNTTO
              })
              .select('Purchasing_Document', 'Item', 'Is_Stock_Transfer', 'Is_Return_Item', 'Is_Header_Deleted', 'Is_Item_Deleted',
                      'Is_Item_Rejected', 'Is_Header_Blocked', 'Is_Item_Blocked', 'Is_Request_Or_Contract', 'Is_Ack_Required',
                      'Has_Conf_Control', 'Has_Ack_Number', 'Is_Delivery_Completed', 'Is_Final_Invoice_No_GR', 'Is_Final_Delivery',
                      'Underdelivery_Pct'))
ITEM_BOOL_FLAGS = [c for c in item_flags.columns if c.startswith(('Is_', 'Has_'))]

# ── Schedule lines (EKET): quantities filled once (a missing quantity counts as 0) ──────────────────────────
_w_item = Window.partitionBy('Purchasing_Document', 'Item')
lines = (eket.select('Purchasing_Document', 'Item', 'Schedule_Line',
                     F.col('Schedule_Line').cast('int').alias('Line_No'),
                     F.col('Delivery_Date').alias('Line_Date'),
                     F.col('Scheduled_Quantity').cast('double').alias('Qty_Scheduled'),
                     F.coalesce(F.col('Quantity_Delivered').cast('double'), F.lit(0.0)).alias('Qty_Received'),
                     F.coalesce(F.col('Qty_Delivered_STO').cast('double'), F.lit(0.0)).alias('Qty_Issued'),
                     F.coalesce(F.col('Qty_Reduced_MRP').cast('double'), F.lit(0.0)).alias('Qty_Reduced'),
                     F.coalesce(F.col('Committed_Quantity').cast('double'), F.lit(0.0)).alias('Qty_Committed'),
                     (F.col('Committed_Date').isNotNull()
                      & ~F.trim(F.col('Committed_Date').cast('string')).isin('', '00000000')).alias('Has_Committed_Date'))
         .join(item_flags, ['Purchasing_Document', 'Item'], 'left')
         .fillna(False, subset=ITEM_BOOL_FLAGS).fillna(0.0, subset=['Underdelivery_Pct'])
         .withColumns({
             # still to receive on the line. DABMG is not deducted: the confirmations allocated below carry that quantity.
             'Line_Open':      F.greatest(F.col('Qty_Scheduled') - F.col('Qty_Received'), F.lit(0.0)),
             # source side: stock transfers only (issued at the supplying plant)
             'Src_Remaining':  F.when(F.col('Is_Stock_Transfer'),
                                      F.greatest(F.col('Qty_Scheduled') - F.col('Qty_Issued') - F.col('Qty_Reduced'), F.lit(0.0)))
                                .otherwise(F.lit(0.0)),
             # C22: what may stay unreceived and still close the line (under-delivery tolerance % of the line, at least OS_TOLERANCE)
             'Tolerance_Qty':  F.greatest(F.lit(OS_TOLERANCE), F.coalesce(F.col('Qty_Scheduled'), F.lit(0.0)) * F.col('Underdelivery_Pct') / 100),
         })
         .withColumns({
             # C21 / C22: closed when nothing (or only the tolerance) is left to receive, or the delivery is marked completed (ELIKZ)
             'Is_Closed_Line':          (F.col('Line_Open') <= F.col('Tolerance_Qty')) | F.col('Is_Delivery_Completed'),
             'Has_Source_Open':         F.col('Src_Remaining') > OS_TOLERANCE,
             'Has_Issued_Not_Received': (F.col('Qty_Issued') - F.col('Qty_Received')) > OS_TOLERANCE,
             'Is_Committed':            (F.col('Qty_Committed') > OS_TOLERANCE) & F.col('Has_Committed_Date'),   # C24: EA EKET_COMMIT
         })
         .withColumn('Alloc_Base', F.when(F.col('Is_Closed_Line'), F.lit(0.0)).otherwise(F.col('Line_Open')))
         .withColumn('Open_Line_Count', F.sum((~F.col('Is_Closed_Line')).cast('int')).over(_w_item))
         .withColumn('Is_Single_Open_Line', F.col('Open_Line_Count') == 1))

# ── Open confirmations (EKES): open quantity = confirmed minus reduced (DABMG); deleted ones are ignored ────────
confs = (ekes.filter(F.trim(F.coalesce(F.col('Confirmation_Deletion'), F.lit(''))) == '')
         .select('Purchasing_Document', 'Item',
                 F.col('Confirmation_Seq').cast('int').alias('Conf_Seq'),
                 F.col('Confirmed_Delivery_Date').alias('Conf_Date'),
                 'Confirmation_Category', 'Inbound_Delivery',
                 F.greatest(F.coalesce(F.col('Confirmation_Quantity').cast('double'), F.lit(0.0))
                            - F.coalesce(F.col('Confirmation_Qty_Reduced_MRP').cast('double'), F.lit(0.0)), F.lit(0.0)).alias('Conf_Open'))
         .filter(F.col('Conf_Open') > OS_TOLERANCE))

# ── C12: cumulative allocation. Lines and confirmations each get an interval [start, end) on the item's running quantity,
#    both in date order (nulls last, then the native sequence); a confirmed piece is the overlap of the two intervals. ──
_run = lambda order: _w_item.orderBy(*order).rowsBetween(Window.unboundedPreceding, Window.currentRow)
lines = (lines.withColumn('_l_end', F.sum('Alloc_Base').over(_run([F.col('Line_Date').asc_nulls_last(), F.col('Line_No')])))
              .withColumn('_l_start', F.col('_l_end') - F.col('Alloc_Base'))
              .withColumn('_item_alloc', F.sum('Alloc_Base').over(_w_item)))
confs = (confs.withColumn('_c_end', F.sum('Conf_Open').over(_run([F.col('Conf_Date').asc_nulls_last(), F.col('Conf_Seq')])))
              .withColumn('_c_start', F.col('_c_end') - F.col('Conf_Open')))

_line_cols = ['Schedule_Line', 'Line_No', 'Is_Closed_Line', 'Has_Source_Open', 'Has_Issued_Not_Received', 'Is_Committed',
              'Is_Single_Open_Line']
pieces = (confs.join(lines.filter(F.col('Alloc_Base') > OS_TOLERANCE)
                          .select('Purchasing_Document', 'Item', '_l_start', '_l_end', *_line_cols),
                     ['Purchasing_Document', 'Item'], 'inner')
          .withColumn('Piece_Qty', F.least('_c_end', '_l_end') - F.greatest('_c_start', '_l_start'))
          .filter(F.col('Piece_Qty') > OS_TOLERANCE)
          .withColumn('Is_Over_Confirmed', F.lit(False)))

# confirmed quantity beyond everything still open on the item's lines: a piece with no schedule line (DQ warning below)
_item_alloc = lines.groupBy('Purchasing_Document', 'Item').agg(F.max('_item_alloc').alias('_item_alloc'))
over = (confs.join(_item_alloc, ['Purchasing_Document', 'Item'], 'left')
             .withColumn('_item_alloc', F.coalesce(F.col('_item_alloc'), F.lit(0.0)))
             .withColumn('Piece_Qty', F.col('_c_end') - F.greatest(F.col('_c_start'), F.col('_item_alloc')))
             .filter(F.col('Piece_Qty') > OS_TOLERANCE)
             .join(lines.groupBy('Purchasing_Document', 'Item').agg(F.max('Is_Single_Open_Line').alias('Is_Single_Open_Line')),
                   ['Purchasing_Document', 'Item'], 'left')
             .withColumns({'Schedule_Line': F.lit(None).cast(eket.schema['Schedule_Line'].dataType),
                           'Line_No': F.lit(None).cast('int'), 'Is_Closed_Line': F.lit(False), 'Has_Source_Open': F.lit(False),
                           'Has_Issued_Not_Received': F.lit(False), 'Is_Committed': F.lit(False),
                           'Is_Single_Open_Line': F.coalesce(F.col('Is_Single_Open_Line'), F.lit(False)),
                           'Is_Over_Confirmed': F.lit(True)}))

_piece_cols = ['Purchasing_Document', 'Item', 'Conf_Seq', 'Conf_Date', 'Confirmation_Category', 'Inbound_Delivery', 'Piece_Qty',
               'Is_Over_Confirmed', *_line_cols]
seg_pieces = (pieces.select(*_piece_cols).unionByName(over.select(*_piece_cols))
              .select('Purchasing_Document', 'Item',
                      F.lit('EKES').alias('Source_Type'),
                      F.col('Conf_Seq').alias('Source_Sequence'),
                      F.col('Conf_Date').alias('Segment_Date'),
                      F.col('Piece_Qty').alias('Segment_Qty'),
                      'Confirmation_Category', 'Inbound_Delivery', 'Is_Over_Confirmed', *_line_cols))

# ── Segments: each schedule line (its unconfirmed remainder) plus each confirmed piece ──────────────────────
_allocated = (pieces.groupBy('Purchasing_Document', 'Item', 'Schedule_Line').agg(F.sum('Piece_Qty').alias('_allocated')))
seg_lines = (lines.join(_allocated, ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')
             .withColumn('Segment_Qty', F.greatest(F.col('Alloc_Base') - F.coalesce(F.col('_allocated'), F.lit(0.0)), F.lit(0.0)))
             .select('Purchasing_Document', 'Item',
                     F.lit('EKET').alias('Source_Type'),
                     F.col('Line_No').alias('Source_Sequence'),
                     F.col('Line_Date').alias('Segment_Date'),
                     'Segment_Qty',
                     F.lit(None).cast('string').alias('Confirmation_Category'),
                     F.lit(None).cast('string').alias('Inbound_Delivery'),
                     F.lit(False).alias('Is_Over_Confirmed'),
                     # an open line whose whole remainder is confirmed: its pieces carry the status (no rule for the empty rest)
                     ((~F.col('Is_Closed_Line')) & F.col('_allocated').isNotNull() & (F.col('Segment_Qty') <= OS_TOLERANCE)
                      & ~F.col('Has_Source_Open')).alias('Is_Fully_Allocated'),
                     *_line_cols))

seg = (seg_lines.unionByName(seg_pieces.withColumn('Is_Fully_Allocated', F.lit(False)))
       .join(item_flags, ['Purchasing_Document', 'Item'], 'left')
       .fillna(False, subset=ITEM_BOOL_FLAGS)
       .withColumns({
           'Is_Open':            (~F.col('Is_Closed_Line') & (F.col('Segment_Qty') > OS_TOLERANCE)) | F.col('Has_Source_Open'),
           # item-level evidence: the item's only open line (and its pieces), and closed lines
           'Uses_Item_Evidence': F.col('Is_Single_Open_Line') | F.col('Is_Closed_Line'),
       }))

# Split number over the open segments of each item: delivery date (nulls last), EKES before EKET on ties, native sequence
# (Jeremy, section 4; question C11). Regenerated on every run; closed lines and fully allocated remainders carry no number.
w_open = (Window.partitionBy('Purchasing_Document', 'Item', 'Is_Open')
          .orderBy(F.col('Segment_Date').asc_nulls_last(), F.when(F.col('Source_Type') == 'EKES', 0).otherwise(1),
                   F.col('Source_Sequence'), F.col('Line_No').asc_nulls_last()))
seg = (seg.withColumn('Row_In_Item', F.row_number().over(w_open))
          .withColumn('Split_Number', F.when(F.col('Is_Open'), F.col('Row_In_Item')))
          .withColumn('Segment_Quantity', F.when(F.col('Is_Closed_Line'), F.lit(0.0)).otherwise(F.col('Segment_Qty')).cast('decimal(18,3)'))
          .drop('Row_In_Item'))

_n_over = seg_pieces.filter('Is_Over_Confirmed').count()
print(f"Segments: {lines.count():,} schedule lines, {seg_pieces.count():,} confirmed pieces ({_n_over:,} over-confirmed)")
if _n_over:
    dq_record('GOLD', 'Fact_PO_LineItem_Schedule', 'over_confirmed', 'WARN',
              f'{_n_over:,} confirmed pieces exceed the open quantity of their item (kept with no schedule line)', observed=_n_over)

In [ ]:
# ▶▶ NEW 20-Oct-2026 · Order_Status_Purchase · step 3: evidence flags (EKBE, EKES, SD Silver LIPS) per segment ◀◀
# Evidence grains (Jeremy, section 5): EKBE exact on PO + item + ETENS for a confirmed piece; EKBE item level for the item's
# only open line (and its pieces) and for closed lines (Uses_Item_Evidence). Several open lines: item-wide goods-issue
# evidence is not copied to each one.
# ▶ CHANGED 6 Nov 2026 (Jeremy, C16 / C17): movement 107 is an advance shipping notice, not a block. A 107 not yet released
# (107 minus 109 above 0) is now shipping-notification evidence (rule 11); rule 6 is retired. EKES categories are read
# from each piece's own confirmation only (the allocation puts every confirmed quantity on its line).

# ── EKBE history: one flag per movement kind, then the quantity and the date of each kind ───────────────────
ekbe_rows = (ekbe.select('Purchasing_Document', 'Item', F.col('Confirmation_Seq').cast('int').alias('Seq'), 'Posting_Date',
                         'Movement_Type', 'Transevent_Type', 'PO_History_Category', 'DebitCredit_Ind',
                         F.col('Signed_Quantity').cast('double').alias('Qty'))
             .withColumns({
                 'Is_Physical_GR':     F.col('Transevent_Type').eqNullSafe('1') | F.col('PO_History_Category').eqNullSafe('E'),
                 'Is_Invoice_Receipt': F.col('Transevent_Type').eqNullSafe('2') | F.col('Transevent_Type').eqNullSafe('3'),
                 'Is_Debit':           F.col('DebitCredit_Ind').eqNullSafe('S'),
                 'Is_Mvt_103':         F.col('Movement_Type').eqNullSafe('103'),   # GR into blocked stock
                 'Is_Mvt_105':         F.col('Movement_Type').eqNullSafe('105'),   # release from GR blocked stock
                 'Is_Mvt_107':         F.col('Movement_Type').eqNullSafe('107'),   # advance shipping notice (Jeremy, 6 Nov)
                 'Is_Mvt_109':         F.col('Movement_Type').eqNullSafe('109'),   # its release
                 'Is_Service_Entry':   F.col('Transevent_Type').eqNullSafe('9'),
                 'Is_Goods_Issue':     F.col('Transevent_Type').eqNullSafe('6'),   # stock transfers
             })
             .withColumns({
                 'Gr_Qty':   F.when(F.col('Is_Physical_GR'), F.col('Qty')),
                 'Gr_Date':  F.when(F.col('Is_Physical_GR') & F.col('Is_Debit'), F.col('Posting_Date')),
                 'Ir_Qty':   F.when(F.col('Is_Invoice_Receipt'), F.col('Qty')),
                 'Ir_Date':  F.when(F.col('Is_Invoice_Receipt') & F.col('Is_Debit'), F.col('Posting_Date')),
                 'B103_Qty': F.when(F.col('Is_Mvt_103'), F.col('Qty')),
                 'B103_Date': F.when(F.col('Is_Mvt_103') & F.col('Is_Debit'), F.col('Posting_Date')),
                 'R105_Qty': F.when(F.col('Is_Mvt_105'), F.col('Qty')),
                 'B107_Qty': F.when(F.col('Is_Mvt_107'), F.col('Qty')),
                 'R109_Qty': F.when(F.col('Is_Mvt_109'), F.col('Qty')),
                 'Svc_Date': F.when(F.col('Is_Service_Entry'), F.col('Posting_Date')),
                 'Gi_Date':  F.when(F.col('Is_Goods_Issue'), F.col('Posting_Date')),
             }))

# one aggregation by item + ETENS; the item level is derived from it (sums add up, latest dates stay latest)
ev_seq = (ekbe_rows.groupBy('Purchasing_Document', 'Item', 'Seq')
          .agg(F.sum('Gr_Qty').alias('Seg_Gr_Qty'),     F.max('Gr_Date').alias('Seg_Gr_Date'),
               F.sum('Ir_Qty').alias('Seg_Ir_Qty'),     F.max('Ir_Date').alias('Seg_Ir_Date'),
               F.sum('B103_Qty').alias('Seg_B103_Qty'), F.max('B103_Date').alias('Seg_B103_Date'), F.sum('R105_Qty').alias('Seg_R105_Qty'),
               (F.coalesce(F.sum('B107_Qty'), F.lit(0.0)) - F.coalesce(F.sum('R109_Qty'), F.lit(0.0))).alias('Seg_ASN_107_Qty'),
               F.max('Svc_Date').alias('Seg_Svc_Date'), F.max('Gi_Date').alias('Seg_Gi_Date')))

ev_item = (ev_seq.groupBy('Purchasing_Document', 'Item')
           .agg(F.sum('Seg_Gr_Qty').alias('Item_Gr_Qty'),     F.max('Seg_Gr_Date').alias('Item_Gr_Date'),
                F.sum('Seg_Ir_Qty').alias('Item_Ir_Qty'),     F.max('Seg_Ir_Date').alias('Item_Ir_Date'),
                F.max('Seg_B103_Date').alias('Item_B103_Date'),
                (F.coalesce(F.sum('Seg_B103_Qty'), F.lit(0.0)) - F.coalesce(F.sum('Seg_R105_Qty'), F.lit(0.0))).alias('Item_Blocked_103_Qty'),
                F.sum('Seg_ASN_107_Qty').alias('Item_ASN_107_Qty'),
                F.max('Seg_Svc_Date').alias('Item_Svc_Date'), F.max('Seg_Gi_Date').alias('Item_Gi_Date')))

ev_exact = (ev_seq.filter(F.col('Seq') > 0)                                    # history that names an EKES confirmation (ETENS)
            .withColumnRenamed('Seq', 'Source_Sequence')
            .withColumn('Source_Type', F.lit('EKES')))

seg_ev = (seg.join(ev_item,  ['Purchasing_Document', 'Item'], 'left')
             .join(ev_exact, ['Purchasing_Document', 'Item', 'Source_Type', 'Source_Sequence'], 'left'))

# A delivery that references the PO item. SD owns the delivery items (LIPS): looked up in SD Silver ('sd_silver' location in
# NB_Config_PROC), not loaded again. ▶ CHANGED 22-Oct-2026: was a PROC Silver LIPS table loaded from the DAB lakehouse.
if 'sd_silver' in LOC and table_exists('sd_silver', 'LIPS'):
    ev_delivery = (read_table('sd_silver', 'LIPS', active=True)
                   .select(_NZ(F.col('Reference_Document')).alias('Purchasing_Document'),
                           _NZ(F.col('Reference_Item')).alias('Item'))
                   .dropDuplicates().withColumn('Item_Has_Delivery', F.lit(True)))
    seg_ev = seg_ev.join(ev_delivery, ['Purchasing_Document', 'Item'], 'left')
else:
    print('  Note: SD Silver LIPS not reachable (add SD_SILVER_LH_ABFSS to VL_PROC and run SD Silver first): rule 10 uses the EKES inbound delivery reference only')
    seg_ev = seg_ev.withColumn('Item_Has_Delivery', F.lit(False))
seg_ev = seg_ev.fillna(False, subset=['Item_Has_Delivery'])

# ── Evidence flags: the segment's own (exact) evidence first, else the item's when Uses_Item_Evidence ───────
_cat = lambda cats: F.coalesce(F.col('Confirmation_Category').isin(cats), F.lit(False)) if cats else F.lit(False)
seg_ev = seg_ev.withColumns({
    'Has_GR_Date':         F.coalesce(F.col('Seg_Gr_Date'),   F.when(F.col('Uses_Item_Evidence'), F.col('Item_Gr_Date'))).isNotNull(),
    'Has_IR_Date':         F.coalesce(F.col('Seg_Ir_Date'),   F.when(F.col('Uses_Item_Evidence'), F.col('Item_Ir_Date'))).isNotNull(),
    'Has_Item_GR':         F.col('Item_Gr_Date').isNotNull(),                     # any GR on the item (rule 2 blocked-after-GR, rule 1 EGLKZ)
    'Has_Item_GI':         F.col('Item_Gi_Date').isNotNull(),
    # 22-Oct-2026 (EA Support 12 / 14 / 19 Oct): a 103 posting is enough. Jeremy (C17, 6 Nov): OK to flag 103 as GR blocked stock.
    'Has_Blocked_103':     F.coalesce(F.col('Seg_B103_Date'), F.when(F.col('Uses_Item_Evidence'), F.col('Item_B103_Date'))).isNotNull(),
    # 6 Nov 2026: an advance shipping notice (107) not yet released
    'Has_ASN_107':         F.coalesce(F.col('Seg_ASN_107_Qty'), F.when(F.col('Uses_Item_Evidence'), F.col('Item_ASN_107_Qty')), F.lit(0.0)) > OS_TOLERANCE,
    'Has_Service_Entry':   F.coalesce(F.col('Seg_Svc_Date'),  F.when(F.col('Uses_Item_Evidence'), F.col('Item_Svc_Date'))).isNotNull(),
    'Has_Goods_Issue':     F.coalesce(F.col('Seg_Gi_Date'),   F.when(F.col('Uses_Item_Evidence'), F.col('Item_Gi_Date'))).isNotNull(),
    'Has_Net_GR_Or_IR':    (F.coalesce(F.col('Seg_Gr_Qty'), F.when(F.col('Uses_Item_Evidence'), F.col('Item_Gr_Qty')), F.lit(0.0)) > OS_TOLERANCE)
                           | (F.coalesce(F.col('Seg_Ir_Qty'), F.when(F.col('Uses_Item_Evidence'), F.col('Item_Ir_Qty')), F.lit(0.0)) > OS_TOLERANCE),
    'Has_Delivery':        F.col('Item_Has_Delivery'),
    'Has_Inbound_Delivery_Ref': (F.col('Source_Type') == 'EKES') & (F.trim(F.coalesce(F.col('Inbound_Delivery'), F.lit(''))) != ''),
    # EKES confirmation categories (Lincoln's customizing, C1): the piece's own confirmation
    'Has_Receipt_Conf':    _cat(OS_CAT_RECEIPT),
    'Has_Shipping_Conf':   _cat(OS_CAT_SHIPPING),
    'Has_Call_Conf':       _cat(OS_CAT_CALL),
    'Has_Available_Conf':  _cat(OS_CAT_AVAILABLE),
    'Has_Confirmed_Conf':  _cat(OS_CAT_CONFIRMED),
})

In [ ]:
# ▶▶ NEW 20-Oct-2026 · Order_Status_Purchase · step 4: rule flags and the status (first rule that is true wins) ◀◀
# Layer 2: one flag per rule, written only from the fact flags above. Rule 18 (Ordered) has no flag: it is what is left.
# ▶ CHANGED 6 Nov 2026 (Jeremy's feedback, v2): rule 0 Return item (C20); rule 1 adds EA early exits 6 and 8 (C24); rules 2 / 3
# close on the under-delivery tolerance and ELIKZ (C21 / C22), and a line blocked after a goods receipt is Closed by GR (C2);
# rule 6 retired, its 107 now feeds rule 11 (C16 / C17); rule 14 uses EA's committed schedule line (C24, against C19).
seg_flags = seg_ev.withColumns({
    'Is_Return_Item':           F.col('Is_Return_Item'),                                                                               # 0 (item flag)
    'Is_Cancelled':             F.col('Is_Header_Deleted') | F.col('Is_Item_Deleted') | F.col('Is_Item_Rejected')
                                | F.col('Is_Final_Invoice_No_GR')                                                     # EA exit 6
                                | (F.col('Is_Final_Delivery') & ~F.col('Has_Item_GR') & ~F.col('Has_Item_GI')),       # EA exit 8      # 1
    'Is_Closed_By_GR':          (F.col('Is_Closed_Line') & F.col('Has_GR_Date'))
                                | ((F.col('Is_Header_Blocked') | F.col('Is_Item_Blocked')) & F.col('Has_Item_GR')),    # C2             # 2
    'Is_Closed_By_IR':          F.col('Is_Closed_Line'),                                                                               # 3 (also a closed line with no invoice)
    'Is_Blocked':               F.col('Is_Header_Blocked') | F.col('Is_Item_Blocked'),                                                 # 4
    'Is_GR_Blocked':            F.col('Has_Blocked_103'),                                                                              # 5
    'Is_Receipt_From_Supplier': F.col('Has_Receipt_Conf'),                                                                             # 7
    'Is_Service_Entry':         F.col('Has_Service_Entry'),                                                                            # 8
    'Is_Goods_Issued':          F.col('Is_Stock_Transfer')
                                & F.when(F.col('Is_Single_Open_Line'), F.col('Has_Goods_Issue')).otherwise(F.col('Has_Issued_Not_Received')),   # 9
    'Is_Outbound_Delivery':     F.col('Is_Stock_Transfer') & ((F.col('Is_Single_Open_Line') & F.col('Has_Delivery')) | F.col('Has_Inbound_Delivery_Ref')),   # 10
    'Is_Shipping_Notification': F.col('Has_Shipping_Conf') | F.col('Has_ASN_107'),                                                     # 11
    'Is_Call_From_Supplier':    F.col('Has_Call_Conf'),                                                                                # 12
    'Is_Goods_Available':       F.col('Has_Available_Conf'),                                                                           # 13
    'Is_Stock_Committed':       F.col('Is_Stock_Transfer') & F.col('Is_Committed'),                                                    # 14
    'Is_Confirmed':             F.col('Has_Confirmed_Conf') | F.col('Has_Ack_Number'),                                                 # 15
    'Is_To_Be_Ordered':         F.col('Is_Request_Or_Contract'),                                                                       # 16
    'Is_To_Be_Confirmed':       F.col('Is_Ack_Required') | F.col('Has_Conf_Control'),                                                  # 17
})
seg_flags = seg_flags.persist()   # ▶ 17 Nov 2026: read by cells 22 and 23 several times; released at the end of cell 23

# Layer 3: the first true flag, in the order of OS_RULE_FLAGS (rule numbers from OS_RULES, NB_Config_PROC); none true = 18.
# A line whose whole open remainder is confirmed gets no rule of its own (Is_Fully_Allocated): its pieces carry the status.
_rule_nos = [n for n, _ in OS_RULES if n != 18]
assert len(_rule_nos) == len(OS_RULE_FLAGS), 'OS_RULES and OS_RULE_FLAGS (NB_Config_PROC) must list the same rules'
_rule = F.lit(18)
for _n, _f in reversed(list(zip(_rule_nos, OS_RULE_FLAGS))):
    _rule = F.when(F.col(_f), F.lit(_n)).otherwise(_rule)
os_rules = F.broadcast(spark.createDataFrame(OS_RULES, ['Order_Status_Rule', 'Rule_Status']))
seg_status = (seg_flags
    .withColumn('Order_Status_Rule', F.when(~F.col('Is_Fully_Allocated'), _rule))
    .join(os_rules, 'Order_Status_Rule', 'left')
    .withColumn('Order_Status_Purchase',                    # rule 1: partially when goods or invoices were received, else completely
        F.when((F.col('Order_Status_Rule') == 1) & F.col('Has_Net_GR_Or_IR'), 'Cancelled partially')
         .otherwise(F.col('Rule_Status')))
    .select('Purchasing_Document', 'Item', 'Source_Type', 'Source_Sequence', 'Schedule_Line', 'Segment_Date',
            'Split_Number', 'Segment_Quantity', 'Order_Status_Rule', 'Order_Status_Purchase', *OS_RULE_FLAGS,
            'Is_Over_Confirmed', 'Is_Fully_Allocated'))

print('Order status per rule:')
seg_status.groupBy('Order_Status_Rule', 'Order_Status_Purchase').count().orderBy('Order_Status_Rule').show(30, truncate=False)

In [ ]:
# ── Fact_PO_LineItem_Schedule (renamed from Fact_PO_Spend_Analytics, 6 Oct 2026) — one row per schedule line ──
# Grain (17 Nov 2026): Purchasing_Document + Item + Schedule_Line, key PO_Schedule_Status_SK. The status is built per segment
# (EKET remainder and confirmed pieces, cells 19 to 21) and rolled up to the line at the end of this cell; the segments live on in
# Fact_PO_Order_Status_Evidence. 6 Nov – 16 Nov 2026 the fact itself was one row per segment (PO_Segment_SK).

print('\nBuilding Fact_PO_LineItem_Schedule...')

# ── EKPO + EKKO — keys and measures only ─────────────────────────────────────
# PO_SK / PO_Item_SK come with EKET (Silver sk_joins); EKPO / EKKO add measures and their own dimension keys
ekpo_sel = ekpo.select(
    'Purchasing_Document', 'Item',
    'Short_Text', 'Net_Order_Value',
    'Net_Order_Value_USD_EOM', 'Net_Order_Value_USD_FIX',
    'Net_Order_Value_EUR_EOM', 'Net_Order_Value_EUR_FIX',
    'Delivery_Completed', 'Req_Tracking_Number',
    'Material_SK', 'Plant_SK', 'PIR_Purch_Org_SK', 'Material_Plant_SK',   # Material_Plant_SK: team edit, 21 Oct 2026 (EKPO Silver sk_lookup)
    'Address',   # EKPO ADRNR: team edit in Fabric, 3 Nov 2026
    F.col('Material').alias('_sa_material'), F.col('Plant').alias('_sa_plant'),   # MARC lookup, dropped below
)
ekko_sel = ekko.select(
    'Purchasing_Document',
    'Vendor_SK', 'Purch_Org_SK', 'Purchasing_Group_SK',
    'Currency',
    F.col('Document_Date').alias('PO_Document_Date'),
    F.col('Supplier').alias('_sa_supplier'), F.col('Purch_Organization').alias('_sa_purch_org'),   # LFM1 lookup
    'Company_Code', 'Purchasing_Group',   # EA Open PO lines (codes; not on Fact_PO_Delivery_Status)
)

# ── EKBE subqueries — aggregated to item grain ────────────────────────────────
# GR: latest goods receipt posting date per PO item
ekbe_gr = (ekbe
    .filter(F.col('Movement_Type').isin('101', '105'))
    .groupBy('Purchasing_Document', 'Item')
    .agg(F.max('Posting_Date').alias('Realized_Finish_Date')))

# IR: sum of signed invoice receipt amounts per PO item (VGABE='2')
ekbe_ir = (ekbe
    .filter(F.col('Transevent_Type') == '2')
    .groupBy('Purchasing_Document', 'Item')
    .agg(
        F.sum('Signed_Amount_LC'     ).alias('Invoiced_Value_Raw'),
        F.sum('Signed_Amount_USD_EOM').alias('Invoiced_Value_USD_EOM_Raw'),
        F.sum('Signed_Amount_USD_FIX').alias('Invoiced_Value_USD_FIX_Raw'),
        F.sum('Signed_Amount_EUR_EOM').alias('Invoiced_Value_EUR_EOM_Raw'),
        F.sum('Signed_Amount_EUR_FIX').alias('Invoiced_Value_EUR_FIX_Raw'),
    ))

# ── Build fact ────────────────────────────────────────────────────────────────
fact_sa = (eket
    .join(ekpo_sel, ['Purchasing_Document', 'Item'], 'left')
    .join(ekko_sel, 'Purchasing_Document',            'left')
    .join(ekbe_gr,  ['Purchasing_Document', 'Item'],  'left')
    .join(ekbe_ir,  ['Purchasing_Document', 'Item'],  'left')
)

# ── Derived columns ───────────────────────────────────────────────────────────
fact_sa = (fact_sa
    .withColumn('Ordered_Value',         F.col('Net_Order_Value'))
    .withColumn('Ordered_Value_USD_EOM', F.col('Net_Order_Value_USD_EOM'))
    .withColumn('Ordered_Value_USD_FIX', F.col('Net_Order_Value_USD_FIX'))
    .withColumn('Ordered_Value_EUR_EOM', F.col('Net_Order_Value_EUR_EOM'))
    .withColumn('Ordered_Value_EUR_FIX', F.col('Net_Order_Value_EUR_FIX'))
    .withColumn('Invoiced_Value',         F.coalesce(F.col('Invoiced_Value_Raw'),         F.lit(0.0)))
    .withColumn('Invoiced_Value_USD_EOM', F.coalesce(F.col('Invoiced_Value_USD_EOM_Raw'), F.lit(0.0)))
    .withColumn('Invoiced_Value_USD_FIX', F.coalesce(F.col('Invoiced_Value_USD_FIX_Raw'), F.lit(0.0)))
    .withColumn('Invoiced_Value_EUR_EOM', F.coalesce(F.col('Invoiced_Value_EUR_EOM_Raw'), F.lit(0.0)))
    .withColumn('Invoiced_Value_EUR_FIX', F.coalesce(F.col('Invoiced_Value_EUR_FIX_Raw'), F.lit(0.0)))
    .withColumn('Days_Late',
        F.datediff(F.col('Realized_Finish_Date'), F.col('Delivery_Date')))
    .withColumn('Delivery_Reliability',
        F.when(F.col('Realized_Finish_Date').isNull(),                  F.lit('No GR'))
         .when(F.col('Realized_Finish_Date') < F.col('Delivery_Date'),  F.lit('Early'))
         .when(F.col('Realized_Finish_Date') == F.col('Delivery_Date'), F.lit('On Time'))
         .otherwise(                                                      F.lit('Late')))
    # ▶ CHANGED: was .withColumn('Order_Status_Purchase', F.lit('Not Applicable').cast('string')); now set from the segment status below
    .drop('Net_Order_Value',
          'Net_Order_Value_USD_EOM', 'Net_Order_Value_USD_FIX',
          'Net_Order_Value_EUR_EOM', 'Net_Order_Value_EUR_FIX',
          'Invoiced_Value_Raw',
          'Invoiced_Value_USD_EOM_Raw', 'Invoiced_Value_USD_FIX_Raw',
          'Invoiced_Value_EUR_EOM_Raw', 'Invoiced_Value_EUR_FIX_Raw')
)

# ── EA audit reports (phase 1): plant data of the material (MD Silver MARC) and the vendor's purchasing-org
# Incoterms (MD Silver LFM1); both lookups are unique on their keys.
_marc = (read_table('md_silver', 'MARC', active=True)
         .select(_NZ(F.col('Material')).alias('_mc_mat'), _NZ(F.col('Plant')).alias('_mc_plant'), 'MRP_Controller',
                 F.col('Country_Of_Origin').alias('Plant_Country_Of_Origin'), F.col('Profit_Center').alias('Plant_Profit_Center'))
         .dropDuplicates(['_mc_mat', '_mc_plant']))
_lfm1 = (read_table('md_silver', 'LFM1', active=True)
         .select(_NZ(F.col('Supplier')).alias('_lf_sup'), F.trim(F.col('Purch_Organization')).alias('_lf_org'),
                 F.col('Incoterms').alias('Vendor_PO_Incoterms'))
         .dropDuplicates(['_lf_sup', '_lf_org']))
fact_sa = (fact_sa
    .join(_marc, (_NZ(F.col('_sa_material')) == F.col('_mc_mat')) & (_NZ(F.col('_sa_plant')) == F.col('_mc_plant')), 'left')
    .join(_lfm1, (_NZ(F.col('_sa_supplier')) == F.col('_lf_sup')) & (F.trim(F.col('_sa_purch_org')) == F.col('_lf_org')), 'left')
    .drop('_mc_mat', '_mc_plant', '_lf_sup', '_lf_org', '_sa_material', '_sa_plant', '_sa_supplier', '_sa_purch_org'))

# ── EA Open PO lines with stock level (6 Oct 2026): item columns from Fact_PO_Delivery_Status ────────
# Fact_PO_Delivery_Status (PO item grain, written above in this run) already holds these, so they are read from it
# on PO_Item_SK instead of being rebuilt. Item values repeat on each schedule line of the item.
_ds = (gold('Fact_PO_Delivery_Status')
       .select(F.col('PO_Item_SK').alias('_ds_item_sk'),
               'Execution_Status', 'Purch_Doc_Category', 'Order_Quantity', 'Order_Price_Unit', 'Net_Order_Price',
               'Price_Unit', 'Price_Per_Unit', 'Purchasing_Doc_Type', 'Item_Category',
               'Total_Stock', 'Safety_Stock', 'Below_Safety_Stock', 'Acknowledgment_Reqd', 'Created_By',
               'Acct_Assignment_Cat', 'Header_Deletion_Indicator', 'Deletion_Indicator',
               F.col('Created_On').alias('Header_Created_On'),            # EKKO AEDAT
               F.col('Creation_Date').alias('Item_Creation_Date'),        # EKPO CREATIONDATE
               F.col('Confirmation_Control').alias('Conf_Control'))       # EKPO BSTAE
       .dropDuplicates(['_ds_item_sk']))
fact_sa = (fact_sa
    .join(_ds, F.col('PO_Item_SK') == F.col('_ds_item_sk'), 'left')
    .drop('_ds_item_sk')
    .withColumn('Schedule_Line_Open_Quantity',                   # EKET MENGE - WEMNG, per schedule line
                F.col('Scheduled_Quantity') - F.coalesce(F.col('Quantity_Delivered'), F.lit(0)))
    # Qty_Delivered_STO (EKET GLMNG) comes with the EKET base.
    # Not built yet, kept as comments until the rule is there:
    # .withColumn('Confirmed_Quantity', ...)                          # rule to define (EKES confirmations?)
    # .withColumn('GR_Blocked_Quantity', ...)                         # rule to define (EKBE blocked-stock movements?)
    # ▶ CHANGED: Order_Status_Purchase no longer keeps a placeholder; the 18 rules are applied below (rule flags, then Order_Status_Rule; settings in NB_Config_PROC).
)

# EA audit flag (Is_Audit_): purchase-order schedule lines, audit report 11 (deck V4 slide 22). The purchasing orgs
# ('modify as needed' in the deck) are a preset page filter in the report, not part of the flag.
fact_sa = fact_sa.withColumn('Is_Audit_Goods_In_Transit', F.coalesce(F.col('Purch_Doc_Category') == 'F', F.lit(False)))

# ▶▶ NEW 20-Oct-2026 · Order_Status_Purchase · step 5: segment keys, EKES rows and the status ◀◀
# One row per segment: the EKET lines built above plus one row per confirmed piece (▶ CHANGED 6 Nov 2026, C12: an open EKES
# confirmation allocated to a schedule line; it carries that line's Schedule_Line and PO_Schedule_SK). EKES rows carry the
# item's attributes and SKs; the union leaves every EKET measure and item-level repeated value empty on them, so existing sums do not change.
fact_sa = (fact_sa.withColumn('Source_Type', F.lit('EKET'))
                  .withColumn('Source_Sequence', F.col('Schedule_Line').cast('int')))
eket_only = eket.drop('Purchasing_Document', 'Item', 'PO_Item_SK', 'PO_SK').columns       # every EKET column except the keys to the item
item_attrs = (fact_sa.drop(*eket_only, 'Source_Type', 'Source_Sequence',
                           'Realized_Finish_Date', 'Days_Late', 'Delivery_Reliability', 'Schedule_Line_Open_Quantity',
                           'Ordered_Value', 'Ordered_Value_USD_EOM', 'Ordered_Value_USD_FIX', 'Ordered_Value_EUR_EOM', 'Ordered_Value_EUR_FIX',
                           'Invoiced_Value', 'Invoiced_Value_USD_EOM', 'Invoiced_Value_USD_FIX', 'Invoiced_Value_EUR_EOM', 'Invoiced_Value_EUR_FIX',
                           'Order_Quantity', 'Total_Stock', 'Safety_Stock', 'Below_Safety_Stock')
              .dropDuplicates(['Purchasing_Document', 'Item']))
_line_sk = eket.select('Purchasing_Document', 'Item', 'Schedule_Line', 'PO_Schedule_SK').dropDuplicates(['Purchasing_Document', 'Item', 'Schedule_Line'])
es_rows = (seg_pieces.select('Purchasing_Document', 'Item', 'Source_Type', 'Source_Sequence', 'Schedule_Line',
                             F.col('Segment_Date').alias('Delivery_Date'))
           .join(_line_sk, ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')      # over-confirmed pieces have no line
           .join(item_attrs, ['Purchasing_Document', 'Item'], 'left'))
_line_only = [c for c in fact_sa.columns if c not in es_rows.columns]   # EKET measures and line values, empty on EKES rows
fact_sa = fact_sa.unionByName(es_rows, allowMissingColumns=True)
# the status of each segment; Schedule_Line is part of the key (a confirmation can split over two lines) and is empty on
# over-confirmed pieces, so the keys are compared null-safe
_seg_keys = ['Purchasing_Document', 'Item', 'Source_Type', 'Source_Sequence', 'Schedule_Line']
# ▶ 15 Nov 2026: only the final status (number and description) is stored here; the 17 rule flags and the other flags behind
#   it are in Fact_PO_Order_Status_Evidence (cell below). Is_Fully_Allocated is used for the fold below and then dropped.
_st = seg_status.select(*_seg_keys, 'Split_Number', 'Segment_Quantity', 'Order_Status_Rule', 'Order_Status_Purchase',
                        'Is_Fully_Allocated')
_st = _st.select(*[F.col(c).alias(f'_st_{c}') if c in _seg_keys else F.col(c) for c in _st.columns])
fact_sa = (fact_sa.join(_st, reduce(lambda a, b: a & b, [fact_sa[c].eqNullSafe(_st[f'_st_{c}']) for c in _seg_keys]), 'left')
                  .drop(*[f'_st_{c}' for c in _seg_keys]))

# ▶ 15 Nov 2026 (feedback: no rows without a status). An open line whose whole remainder is confirmed (Is_Fully_Allocated) had a
# row of its own with no status; its confirmed pieces carry the status. That row is folded into the line's first confirmed
# piece (by date, then sequence): the piece takes the line's EKET measures and line values, the line row goes. Every EKET line's
# measures still appear on exactly one row, so the sums do not change; the piece keeps its own keys, date and status.
_fa = F.col('Source_Type').eqNullSafe('EKET') & F.coalesce(F.col('Is_Fully_Allocated'), F.lit(False))
_moved = (fact_sa.filter(_fa)
          .select('Purchasing_Document', 'Item', 'Schedule_Line', *[F.col(c).alias(f'_ln_{c}') for c in _line_only])
          .withColumn('_pc_rank', F.lit(1)))
_w_piece = (Window.partitionBy('Purchasing_Document', 'Item', 'Schedule_Line', 'Source_Type')
            .orderBy(F.col('Delivery_Date').asc_nulls_last(), F.col('Source_Sequence')))
fact_sa = (fact_sa.filter(~_fa)
    .withColumn('_pc_rank', F.when(F.col('Source_Type') == 'EKES', F.row_number().over(_w_piece)))
    .join(_moved, ['Purchasing_Document', 'Item', 'Schedule_Line', '_pc_rank'], 'left')
    .withColumns({c: F.coalesce(F.col(f'_ln_{c}'), F.col(c)) for c in _line_only})
    .drop('_pc_rank', 'Is_Fully_Allocated', *[f'_ln_{c}' for c in _line_only]))

# ▶ 17 Nov 2026 (business feedback, Procurment/Fabric_output_V4_20261116.xlsx): one row per schedule line.
#   The segments above gave a partly confirmed line two rows with two statuses (259 keys in the extract: a ZS / GW piece →
#   Shipping notification, the EKET remainder → To be confirmed) and the over-confirmed pieces 4,398 rows with no schedule line.
#   Now: over-confirmed pieces leave the fact (they stay in Fact_PO_Order_Status_Evidence; cell 19 counts them in the
#   over_confirmed warning); each line keeps the status of its segment with the LOWEST rule number (the first rule that is true
#   wins, as inside a segment), tie by date then sequence; its EKET measures sit on that one row (they were on exactly one
#   segment before); Segment_Quantity becomes the line's open quantity over its segments; Piece_Count and
#   Confirmation_Categories say what was rolled up; Status_Segment_Date is the date of the segment that gave the status.
#   Fact_OTD keeps PO_Schedule_SK and is not affected; the pieces remain in the evidence table.
_lk = ['Purchasing_Document', 'Item', 'Schedule_Line']
fact_sa = fact_sa.filter(F.col('Schedule_Line').isNotNull())
_w_line = Window.partitionBy(*_lk).orderBy(F.col('Order_Status_Rule').asc_nulls_last(),
                                           F.col('Delivery_Date').asc_nulls_last(), F.col('Source_Sequence'))
_line_vals = (fact_sa.groupBy(*_lk)
              .agg(*[F.max(c).alias(c) for c in _line_only],                      # each line's measures sit on exactly one segment
                   F.sum('Segment_Quantity').alias('Segment_Quantity'),
                   F.sum((F.col('Source_Type') == 'EKES').cast('int')).alias('Piece_Count')))
_line_cats = (seg_pieces.filter(~F.col('Is_Over_Confirmed')).groupBy(*_lk)
              .agg(F.concat_ws('|', F.sort_array(F.collect_set('Confirmation_Category'))).alias('Confirmation_Categories')))
_line_date = eket.select(*_lk, F.col('Delivery_Date').alias('_line_date')).dropDuplicates(_lk)
fact_sa = (fact_sa.withColumn('_rk', F.row_number().over(_w_line)).filter(F.col('_rk') == 1)
           .drop('_rk', 'Segment_Quantity', *_line_only)
           .withColumnRenamed('Delivery_Date', 'Status_Segment_Date')
           .join(_line_vals, _lk, 'left').join(_line_cats, _lk, 'left').join(_line_date, _lk, 'left')
           .withColumnRenamed('_line_date', 'Delivery_Date')
           .withColumn('Piece_Count', F.coalesce(F.col('Piece_Count'), F.lit(0)))
           .drop('Source_Type', 'Source_Sequence'))

# ── NOT LIVE: proposals from the 17 Nov 2026 feedback review, each waits for the decision named (see PROC/HANDOFF.md) ────
# D2 (business: confirm, incl. the EKPO UNTTO tolerance) — a line-level execution status next to the item's. Execution_Status
#    comes from Fact_PO_Delivery_Status at PO item grain; Every Angle's is per schedule line, so a received line on a partly
#    received item showed Partially Open (9,292 rows in the extract).
# fact_sa = (fact_sa.withColumnRenamed('Execution_Status', 'Item_Execution_Status')
#     .withColumn('Execution_Status',
#         F.when(F.col('Item_Execution_Status').isin('Cancelled_Header', 'Cancelled_Line'), F.col('Item_Execution_Status'))
#          .when((F.col('Delivery_Completed') == 'X') | (F.col('Schedule_Line_Open_Quantity') <= OS_TOLERANCE), 'Closed')   # + UNTTO tolerance
#          .when(F.coalesce(F.col('Quantity_Delivered'), F.lit(0)) > 0, 'Partially Open')
#          .otherwise('Open')))
# D3 (business: confirm) — zero-quantity schedule lines (EKET MENGE 0 carrying the committed quantity MNG02; 27,392 items in the
#    extract, Every Angle hides them) leave the fact and stay in the evidence table:
# _zero = (F.coalesce(F.col('Scheduled_Quantity'), F.lit(0)) == 0) & (F.coalesce(F.col('Quantity_Delivered'), F.lit(0)) == 0)
# fact_sa = fact_sa.filter(~_zero)          # and a dq_record('GOLD', _T, 'zero_qty_lines', 'WARN', ...) with the count
# D4 (business / Jeremy, Open PO v2 is the tiebreaker) — (a) the execution status follows rule 1:
# fact_sa = fact_sa.withColumn('Execution_Status', F.when(F.col('Order_Status_Rule') == 1, 'Cancelled').otherwise(F.col('Execution_Status')))
#    (b) a closed line with no goods receipt and no invoice is a cancellation, not Closed by order creation or IR: in cell 21 add
#    `| (F.col('Is_Delivery_Completed') & ~F.col('Has_Net_GR_Or_IR'))` to Is_Cancelled.
# D5 / D6 (business, after queries Q6 / Q7 of NB_Check_Feedback_V4_PROC) — rule order for stock transfers (Outbound delivery
#    created before Closed by GR?), rule 14 without PSTYP U, latest confirmation over category rank: change the order of
#    OS_RULES / OS_RULE_FLAGS in NB_Config_PROC cell 5 once decided.

# PO_Schedule_Status_SK: one key per schedule line; keys already in the Gold table are kept
_T = 'Fact_PO_LineItem_Schedule'
if table_exists('gold', _T) and 'PO_Schedule_Status_SK' in read_table('gold', _T).columns:
    fact_sa = stable_sk(fact_sa, _T, 'PO_Schedule_Status_SK', _lk)
else:
    fact_sa = fact_sa.withColumn('PO_Schedule_Status_SK', F.row_number().over(Window.orderBy(*_lk)).cast('int'))

# ── Column ordering: SKs → PKs → dates → measures → status → context ─────────
sk_cols   = [c for c in ('PO_Schedule_Status_SK',   # ▶ 17 Nov 2026: one key per schedule line (was PO_Segment_SK)
                          'PO_Schedule_SK', 'PO_Item_SK', 'PO_SK',
                          'Material_SK', 'Plant_SK', 'Vendor_SK',
                          'Purch_Org_SK', 'Purchasing_Group_SK', 'PIR_Purch_Org_SK')
             if c in fact_sa.columns]
pk_cols   = ['Purchasing_Document', 'Item', 'Schedule_Line', 'Split_Number']   # ▶ 17 Nov 2026: Source_Type / Source_Sequence are on the evidence table
date_cols = [c for c in ('Stat_Rel_Del_Date', 'Delivery_Date', 'Status_Segment_Date', 'Purchase_Order_Date',
                          'Realized_Finish_Date', 'PO_Document_Date')
             if c in fact_sa.columns]
meas_cols = [c for c in (
    'Segment_Quantity', 'Piece_Count',   # ▶ 17 Nov 2026: the line's open quantity over its segments; confirmed pieces rolled up
    'Scheduled_Quantity', 'Scheduled_Quantity_KG', 'Scheduled_Quantity_LB',
    'Quantity_Delivered', 'Qty_Delivered_STO', 'Issued_Quantity',
    'Ordered_Value',
    'Ordered_Value_USD_EOM', 'Ordered_Value_USD_FIX',
    'Ordered_Value_EUR_EOM', 'Ordered_Value_EUR_FIX',
    'Invoiced_Value',
    'Invoiced_Value_USD_EOM', 'Invoiced_Value_USD_FIX',
    'Invoiced_Value_EUR_EOM', 'Invoiced_Value_EUR_FIX',
    'Days_Late',
) if c in fact_sa.columns]
status_cols = [c for c in ('Delivery_Reliability', 'Delivery_Completed',
                            'Order_Status_Purchase',
                            'Order_Status_Rule', 'Confirmation_Categories')    # the rule flags are in Fact_PO_Order_Status_Evidence (15 Nov 2026)
               if c in fact_sa.columns]
ctx_cols    = [c for c in ('Short_Text', 'Currency', 'Address') if c in fact_sa.columns]

placed    = set(sk_cols + pk_cols + date_cols + meas_cols + status_cols + ctx_cols)
rest_cols = [c for c in fact_sa.columns if c not in placed and not c.startswith('_')]
meta_cols = [c for c in fact_sa.columns if c.startswith('_')]

fact_sa = fact_sa.select(sk_cols + pk_cols + date_cols + meas_cols + status_cols + ctx_cols + rest_cols + meta_cols)

# one row per schedule line, rebuilt every run (17 Nov 2026; 6 Nov – 16 Nov one row per segment, PO_Segment_SK)
n = write_gold(fact_sa, 'Fact_PO_LineItem_Schedule', 'PO_Schedule_Status_SK', mode='overwrite')

# Warn only: every row has a status; the EKET measures add up to Silver; one row per schedule line (17 Nov 2026).
_no_status, _qty, _rows, _lines = gold(_T).agg(F.sum(F.col('Order_Status_Rule').isNull().cast('int')), F.sum('Scheduled_Quantity'),
                                              F.count('*'), F.countDistinct(*_lk)).first()
_qty_silver = eket.agg(F.sum('Scheduled_Quantity')).first()[0]
_qty_ok = abs(float(_qty or 0) - float(_qty_silver or 0)) <= OS_TOLERANCE
print(f'Rows without a status: {_no_status or 0:,} (expected 0); Scheduled_Quantity: fact {_qty} | Silver EKET {_qty_silver}'
      f" | {'OK' if _qty_ok else 'DIFFERENT'}")
dq_record('GOLD', _T, 'no_status', 'WARN' if _no_status else 'PASS', f'{_no_status or 0:,} rows without an order status',
          observed=_no_status or 0)
dq_record('GOLD', _T, 'scheduled_qty_vs_silver', 'PASS' if _qty_ok else 'WARN',
          f'Scheduled_Quantity fact {_qty} vs Silver EKET {_qty_silver}')
dq_record('GOLD', _T, 'grain', 'PASS' if _rows == _lines else 'WARN',
          f'{_rows:,} rows for {_lines:,} schedule lines (one row per Purchasing_Document + Item + Schedule_Line expected)',
          observed=_rows, expected=_lines)


In [ ]:
# ── Fact_PO_Order_Status_Evidence (9 Nov 2026): why each segment got its status, for review in SQL ───────────────
# One row per segment (PO_Segment_SK), the grain of cells 19 to 21. ▶ 17 Nov 2026: the fact is one row per schedule line, so a
# line's pieces all carry its PO_Schedule_Status_SK (NULL on over-confirmed pieces, which have no fact row); the status here is the
# segment's own, the fact shows the line's lowest rule. It keeps what cells 19 to 21 compute and then drop: the SAP
# source fields, the line and confirmation quantities, the EKBE evidence (the segment's own and the item's), the evidence
# flags, and the rule flags with the status. Since 15 Nov 2026 the rule flags live only here (the fact keeps the status). So the status of a row can be checked from Gold alone
# (PROC/SQL_Order_Status_Sample.sql). It is for review only: reports read Fact_PO_LineItem_Schedule, and the semantic model
# does not relate the two facts.
print('\nBuilding Fact_PO_Order_Status_Evidence...')

# SAP source fields of the item and its header, under their Silver names (cell 19 keeps only the flags made from them)
_ev_src = (ekpo.select('Purchasing_Document', 'Item',
                       F.col('Deletion_Indicator').alias('Item_Deletion'),                          # LOEKZ
                       'Rejection_Indicator', 'Acknowledgment_Reqd', 'Confirmation_Control',        # ABSKZ, KZABS, BSTAE
                       'Order_Acknowledgment', 'Item_Category_Ext', 'Returns_Item',                 # LABNR, PSTYP (EPSTP), RETPO
                       'Delivery_Completed', 'Underdelivery_Tolerance', 'Invoice_Receipt_Ind',      # ELIKZ, UNTTO, REPOS
                       'Goods_Receipt_Ind', 'Final_Invoice', 'Final_Delivery')                      # WEPOS, EREKZ, EGLKZ
           .dropDuplicates(['Purchasing_Document', 'Item'])
           .join(po_header, 'Purchasing_Document', 'left'))                                         # EKKO LOEKZ, BSTYP

# Schedule line quantities (EKET) and what cell 19 derives from them; the line of a confirmed piece is its allocated line
_ev_line = lines.select('Purchasing_Document', 'Item', 'Schedule_Line', 'Line_Date', 'Qty_Scheduled', 'Qty_Received',
                        'Qty_Issued', 'Qty_Reduced', 'Qty_Committed', 'Has_Committed_Date', 'Line_Open', 'Src_Remaining',
                        'Tolerance_Qty', 'Alloc_Base', 'Open_Line_Count')

# The confirmation of an EKES row (EKES EINDT, MENGE, DABMG)
_ev_conf = (ekes.select(F.col('Purchasing_Document').alias('_cf_doc'), F.col('Item').alias('_cf_item'),
                        F.col('Confirmation_Seq').cast('int').alias('_cf_seq'),
                        'Confirmed_Delivery_Date', 'Confirmation_Quantity', 'Confirmation_Qty_Reduced_MRP')
            .dropDuplicates(['_cf_doc', '_cf_item', '_cf_seq']))

_ev_keys = ['Purchasing_Document', 'Item', 'Source_Type', 'Source_Sequence', 'Schedule_Line']
# ▶ 15 Nov 2026: the row of a fully confirmed line is folded into its first piece in the fact (cell above), so it is left out
#   here too; its pieces show Is_Line_Fully_Confirmed and carry the line's quantities (_ev_line).
_ev_fa = (F.col('Source_Type') == 'EKET') & F.col('Is_Fully_Allocated')
_ev_fa_lines = (seg_flags.filter(_ev_fa).select('Purchasing_Document', 'Item', 'Schedule_Line')
                .withColumn('Is_Line_Fully_Confirmed', F.lit(True)))
evidence = (seg_flags.filter(~_ev_fa).drop('Is_Fully_Allocated')
    .join(_ev_fa_lines, ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')
    .fillna(False, subset=['Is_Line_Fully_Confirmed'])
    .join(_ev_line, ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')         # over-confirmed pieces have no line
    .join(_ev_conf, (F.col('Source_Type') == 'EKES') & (F.col('Purchasing_Document') == F.col('_cf_doc'))
                    & (F.col('Item') == F.col('_cf_item')) & (F.col('Source_Sequence') == F.col('_cf_seq')), 'left')
    .drop('_cf_doc', '_cf_item', '_cf_seq')
    .join(_ev_src, ['Purchasing_Document', 'Item'], 'left'))

# ▶ 17 Nov 2026: the segment's own status (seg_status, cell 21; null-safe on the keys: Schedule_Line is empty on over-confirmed
#   pieces), the keys of the line's fact row and of the item, and the evidence table's own key PO_Segment_SK
_ev_st = seg_status.select(*[F.col(c).alias(f'_f_{c}') for c in _ev_keys], 'Order_Status_Rule', 'Order_Status_Purchase')
evidence = (evidence.join(_ev_st, reduce(lambda a, b: a & b, [evidence[c].eqNullSafe(_ev_st[f'_f_{c}']) for c in _ev_keys]), 'left')
            .drop(*[f'_f_{c}' for c in _ev_keys]))
_ev_line_sk = (gold('Fact_PO_LineItem_Schedule').select('Purchasing_Document', 'Item', 'Schedule_Line', 'PO_Schedule_Status_SK', 'PO_Schedule_SK')
               .dropDuplicates(['Purchasing_Document', 'Item', 'Schedule_Line']))
_ev_item_sk = eket.select('Purchasing_Document', 'Item', 'PO_Item_SK', 'PO_SK').dropDuplicates(['Purchasing_Document', 'Item'])
evidence = (evidence.join(_ev_line_sk, ['Purchasing_Document', 'Item', 'Schedule_Line'], 'left')
                    .join(_ev_item_sk, ['Purchasing_Document', 'Item'], 'left'))
_E = 'Fact_PO_Order_Status_Evidence'
if table_exists('gold', _E) and 'PO_Segment_SK' in read_table('gold', _E).columns:
    evidence = stable_sk(evidence, _E, 'PO_Segment_SK', _ev_keys)
else:
    evidence = evidence.withColumn('PO_Segment_SK', F.row_number().over(Window.orderBy(*_ev_keys)).cast('int'))

# Column order: keys, the status and its rule flags, then the inputs from the source to the evidence flags
_ev_sk     = ['PO_Segment_SK', 'PO_Schedule_Status_SK', 'PO_Schedule_SK', 'PO_Item_SK', 'PO_SK']
_ev_pk     = ['Purchasing_Document', 'Item', 'Schedule_Line', 'Source_Type', 'Source_Sequence', 'Split_Number']
_ev_status = ['Order_Status_Rule', 'Order_Status_Purchase', *OS_RULE_FLAGS]
_ev_src_cols = ['Header_Deletion', 'Header_Category', 'Item_Deletion', 'Rejection_Indicator', 'Item_Category_Ext', 'Returns_Item',
                'Delivery_Completed', 'Underdelivery_Tolerance', 'Invoice_Receipt_Ind', 'Goods_Receipt_Ind', 'Final_Invoice',
                'Final_Delivery', 'Acknowledgment_Reqd', 'Confirmation_Control', 'Order_Acknowledgment',
                'Confirmation_Category', 'Inbound_Delivery', 'Confirmed_Delivery_Date', 'Confirmation_Quantity',
                'Confirmation_Qty_Reduced_MRP']
_ev_rest   = [c for c in evidence.columns if c not in _ev_sk + _ev_pk + _ev_status + _ev_src_cols and not c.startswith('_')]
evidence   = evidence.select(*_ev_sk, *_ev_pk, *_ev_status, *_ev_src_cols, *_ev_rest)

# Grain: one row per segment; every segment with a schedule line points at its fact row; the fact has one row per such line
_ev_n, _ev_d, _ev_nosk, _ev_lines = evidence.agg(
    F.count('*'), F.countDistinct('PO_Segment_SK'),
    F.sum((F.col('Schedule_Line').isNotNull() & F.col('PO_Schedule_Status_SK').isNull()).cast('int')),
    F.countDistinct('Purchasing_Document', 'Item', 'Schedule_Line')).first()        # rows with an empty Schedule_Line are not counted
_fact_n = gold('Fact_PO_LineItem_Schedule').count()
assert _ev_n == _ev_d and not _ev_nosk and _ev_lines == _fact_n, \
    f'Fact_PO_Order_Status_Evidence: {_ev_n:,} rows, {_ev_d:,} segment keys, {_ev_nosk or 0:,} lines without a fact key; ' \
    f'{_ev_lines:,} schedule lines vs {_fact_n:,} fact rows'

n = write_gold(evidence, 'Fact_PO_Order_Status_Evidence', 'PO_Segment_SK', mode='overwrite')
seg_flags.unpersist()   # ▶ 17 Nov 2026: persisted in cell 21

### The cell below contains a draft for the **Fact_OTD** table – <u>it is currently frozen!</u>

In [ ]:
# ============================================================
# FACT_OTD
# ============================================================
# Build dedicated Supplier OTD fact using:
# Fact_PO_Receipts + Fact_PO_LineItem_Schedule

print("\nBuilding Fact_OTD...")

from pyspark.sql.window import Window


# ============================================================
# SOURCE TABLES
# ============================================================

# Load physical receipt history (EKBE-based Gold fact)
receipts = (
    gold("Fact_PO_Receipts")
    .filter(
        (F.col("Transevent_Type") == "1")
        & F.col("Movement_Type").isin("101", "102", "105", "106", "109")
    )
    .select(
        "PO_History_SK",
        "PO_Item_SK",
        "PO_SK",
        "Material_SK",
        "Plant_SK",
        "Purchasing_Document",
        "Item",
        "Posting_Date",
        "Movement_Type",
        "DebitCredit_Ind",
        "Signed_Quantity",
    )
)

# Load planned supplier delivery commitments
schedule = (
    gold("Fact_PO_LineItem_Schedule")
    .select(
        "PO_Schedule_SK",
        "PO_Item_SK",
        "PO_SK",
        "Vendor_SK",
        "Material_SK",
        "Plant_SK",
        "Purch_Org_SK",
        "Purchasing_Group_SK",
        "Purchasing_Document",
        "Item",
        "Schedule_Line",
        "Delivery_Date",
        "Scheduled_Quantity",
    )
)


# ============================================================
# SHARED WINDOWS
# ============================================================

# Reused for:
# - positive receipt ranges
# - negative receipt ranges
# - adjusted receipt allocation ranges
newest_first = (
    Window
    .partitionBy(
        "Purchasing_Document",
        "Item",
    )
    .orderBy(
        F.col("Posting_Date").desc(),
        F.col("PO_History_SK").desc(),
    )
)

schedule_newest_first = (
    Window
    .partitionBy(
        "Purchasing_Document",
        "Item",
    )
    .orderBy(
        F.col("Delivery_Date").desc(),
        F.col("Schedule_Line").desc(),
    )
)


# ============================================================
# STEP 1
# POSITIVE / NEGATIVE RECEIPTS
# ============================================================

# Separate valid receipts from reversal transactions
gr_positive = (
    receipts
    .filter(F.col("Signed_Quantity") > 0)
)

gr_negative = (
    receipts
    .filter(F.col("Signed_Quantity") < 0)
    .withColumn(
        "Reverse_Qty",
        F.abs(F.col("Signed_Quantity")),
    )
)


# ============================================================
# STEP 2
# POSITIVE RECEIPT RANGES
# ============================================================

# Build cumulative quantity ranges for positive receipts
gr_positive = (
    gr_positive
    .withColumn(
        "Pos_End",
        F.sum("Signed_Quantity").over(newest_first),
    )
    .withColumn(
        "Pos_Start",
        F.col("Pos_End") - F.col("Signed_Quantity"),
    )
)


# ============================================================
# STEP 3
# NEGATIVE RECEIPT RANGES
# ============================================================

# Build cumulative quantity ranges for reversals
gr_negative = (
    gr_negative
    .withColumn(
        "Neg_End",
        F.sum("Reverse_Qty").over(newest_first),
    )
    .withColumn(
        "Neg_Start",
        F.col("Neg_End") - F.col("Reverse_Qty"),
    )
)


# ============================================================
# STEP 4
# PALANTIR REVERSAL REDISTRIBUTION
# ============================================================

# Reversals consume newest valid receipts first (LIFO)
reverse_overlap = (
    gr_positive.alias("p")
    .join(
        gr_negative.alias("n"),
        (
            (F.col("p.Purchasing_Document") == F.col("n.Purchasing_Document"))
            & (F.col("p.Item") == F.col("n.Item"))
            & (F.col("p.Pos_Start") < F.col("n.Neg_End"))
            & (F.col("p.Pos_End") > F.col("n.Neg_Start"))
        ),
        "inner",
    )
    .withColumn(
        "Reversed_Qty",
        F.least(
            F.col("p.Pos_End"),
            F.col("n.Neg_End"),
        )
        - F.greatest(
            F.col("p.Pos_Start"),
            F.col("n.Neg_Start"),
        ),
    )
)

# Aggregate reversal quantities per receipt
receipt_adjustments = (
    reverse_overlap
    .groupBy(
        F.col("p.PO_History_SK").alias("PO_History_SK"),
    )
    .agg(
        F.sum("Reversed_Qty").alias("Total_Reversed"),
    )
)

# Calculate net receipt quantity after reversals
otd_ekbe_per_document_date = (
    gr_positive
    .join(
        receipt_adjustments,
        "PO_History_SK",
        "left",
    )
    .withColumn(
        "Receipt_Qty_Adjusted",
        F.greatest(
            F.col("Signed_Quantity")
            - F.coalesce(
                F.col("Total_Reversed"),
                F.lit(0),
            ),
            F.lit(0),
        ),
    )
    .filter(F.col("Receipt_Qty_Adjusted") > 0)
)


# ============================================================
# STEP 5
# RECEIPT LIFO RANGES
# ============================================================

# Build adjusted receipt quantity ranges used for allocation
receipt_lifo = (
    otd_ekbe_per_document_date
    .withColumn(
        "Receipt_End",
        F.sum("Receipt_Qty_Adjusted").over(newest_first),
    )
    .withColumn(
        "Receipt_Start",
        F.col("Receipt_End") - F.col("Receipt_Qty_Adjusted"),
    )
)


# ============================================================
# STEP 6
# SCHEDULE LIFO RANGES
# ============================================================

# Convert schedule commitments into cumulative quantity ranges
schedule_lifo = (
    schedule
    .withColumn(
        "Schedule_End",
        F.sum("Scheduled_Quantity").over(schedule_newest_first),
    )
    .withColumn(
        "Schedule_Start",
        F.col("Schedule_End") - F.col("Scheduled_Quantity"),
    )
)


# ============================================================
# STEP 7
# RECEIPT TO SCHEDULE ALLOCATION
# ============================================================

# Allocate receipt quantities to schedule commitments
allocation = (
    receipt_lifo.alias("r")
    .join(
        schedule_lifo.alias("s"),
        (
            (F.col("r.Purchasing_Document") == F.col("s.Purchasing_Document"))
            & (F.col("r.Item") == F.col("s.Item"))
            & (F.col("r.Receipt_Start") < F.col("s.Schedule_End"))
            & (F.col("r.Receipt_End") > F.col("s.Schedule_Start"))
        ),
        "inner",
    )
    .withColumn(
        "Allocated_Qty",
        F.least(
            F.col("r.Receipt_End"),
            F.col("s.Schedule_End"),
        )
        - F.greatest(
            F.col("r.Receipt_Start"),
            F.col("s.Schedule_Start"),
        ),
    )
    .filter(F.col("Allocated_Qty") > 0)
)


# ============================================================
# STEP 8
# OTD KPI
# ============================================================

# Calculate OTD metrics using planned vs actual dates
fact_otd = (
    allocation
    .withColumn(
        "Days_Early_Late",
        F.datediff(
            F.col("r.Posting_Date"),
            F.col("s.Delivery_Date"),
        ),
    )
    .withColumn(
        "Days_Early",
        F.when(
            F.col("Days_Early_Late") < 0,
            F.abs(F.col("Days_Early_Late")),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "Days_Late",
        F.when(
            F.col("Days_Early_Late") > 0,
            F.col("Days_Early_Late"),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "OTD_Flag",
        F.when(
            (F.col("Days_Early_Late") >= -7)
            & (F.col("Days_Early_Late") <= 3),
            F.lit(1),
        ).otherwise(F.lit(0)),
    )
    .withColumn(
        "OTD_Bucket",
        F.when(
            F.col("Days_Early_Late") < -30,
            F.lit("Early >30"),
        )
        .when(
            F.col("Days_Early_Late") < -7,
            F.lit("Early"),
        )
        .when(
            F.col("Days_Early_Late") <= 3,
            F.lit("On Time"),
        )
        .when(
            F.col("Days_Early_Late") <= 30,
            F.lit("Late"),
        )
        .otherwise(F.lit("Late >30")),
    )
)


# ============================================================
# STEP 9
# FINAL FACT_OTD OUTPUT
# ============================================================

# Select final analytical grain:
# one allocated receipt quantity per schedule line
fact_otd = (
    fact_otd
    .select(
        F.col("r.PO_History_SK").alias("PO_History_SK"),
        F.col("s.PO_Schedule_SK").alias("PO_Schedule_SK"),
        F.col("s.PO_Item_SK").alias("PO_Item_SK"),
        F.col("s.PO_SK").alias("PO_SK"),
        F.col("s.Vendor_SK").alias("Vendor_SK"),
        F.col("s.Material_SK").alias("Material_SK"),
        F.col("s.Plant_SK").alias("Plant_SK"),
        F.col("s.Purch_Org_SK").alias("Purch_Org_SK"),
        F.col("s.Purchasing_Group_SK").alias("Purchasing_Group_SK"),
        F.col("s.Purchasing_Document").alias("Purchasing_Document"),
        F.col("s.Item").alias("Item"),
        F.col("s.Schedule_Line").alias("Schedule_Line"),
        F.col("s.Delivery_Date").alias("Schedule_Date"),
        F.col("r.Posting_Date").alias("Receipt_Date"),
        F.col("Allocated_Qty"),
        F.col("Days_Early_Late"),
        F.col("Days_Early"),
        F.col("Days_Late"),
        F.col("OTD_Flag"),
        F.col("OTD_Bucket"),
    )
)


# ============================================================
# STEP 10
# CREATE STABLE GOLD SURROGATE KEY
# ============================================================

# Fact_OTD has no surrogate key inherited from Silver.
# The combination below represents the allocation grain:
# one receipt history record allocated to one schedule line.
fact_otd = stable_sk(
    fact_otd,
    "Fact_OTD",
    "OTD_SK",
    [
        "PO_History_SK",
        "PO_Schedule_SK",
    ],
)


# ============================================================
# STEP 11
# STANDARD COLUMN ORDER
# ============================================================

# Follow the standard Gold fact layout:
# surrogate keys first, business keys second, remaining columns last.
sk_cols = [
    c
    for c in (
        "OTD_SK",
        "PO_History_SK",
        "PO_Schedule_SK",
        "PO_Item_SK",
        "PO_SK",
        "Vendor_SK",
        "Material_SK",
        "Plant_SK",
        "Purch_Org_SK",
        "Purchasing_Group_SK",
    )
    if c in fact_otd.columns
]

pk_cols = [
    c
    for c in (
        "Purchasing_Document",
        "Item",
        "Schedule_Line",
    )
    if c in fact_otd.columns
]

rest_cols = [
    c
    for c in fact_otd.columns
    if c not in sk_cols + pk_cols
]

fact_otd = fact_otd.select(
    sk_cols + pk_cols + rest_cols
)


# ============================================================
# STEP 12
# WRITE GOLD TABLE
# ============================================================

# Persist final Fact_OTD table into the Gold layer.
# ▶ 18 Nov 2026: overwrite instead of merge. The allocation is rebuilt in full on every run; with merge, an allocation that no
#   longer exists (a receipt reversed, a schedule line changed) stayed in the table and inflated Received_Quantity on
#   Fact_PO_LineItem_Schedule. Fact_OTD now also feeds SM_AUDIT (delivery reliability, first goods receipt date).
n = write_gold(
    fact_otd,
    "Fact_OTD",
    "OTD_SK",
    mode="overwrite",
)

print(f"Fact_OTD rows built: {n}")

In [ ]:
# ── Dim_Source_List (EORD) moved to MD Gold (13 Oct 2026) ──────────────────────────────────────
# EORD is owned by Master Data now (NB_Config_MD), and NB_Gold_MD writes Dim_Source_List with the other MD dimensions.
# Reports and the semantic model read it from MD Gold.

**Audit enhancements, phase 2** (EA audit reports 09 PO Release Strategy, 11 PO Schedule Lines, 14 PPV, 15, 16). Field list: `Audit/List of fields - Audit.xlsx`. Built 26 Oct 2026, switched on 1 Nov 2026, **revised 18 Nov 2026 after the business feedback** (`Procurment/Copy of PROC fields in Audit.xlsx`; mapping `Audit/Audit_Build_Mapping_v4.xlsx`, tab `Feedback 18 Nov`).

The four cells below each re-read their table(s) from Gold, add the columns and re-write in `overwrite` mode, so they run after the whole notebook above (cell 4 needs the `Fact_OTD` cell) and can be re-run on their own. Cells 3 and 4 need `_rc_item` from cell 1 in the same session.

| Cell | Adds | Table(s) |
|---|---|---|
| 1 | `Received_Value_DC` (was `Received_Value_LC`), `GR_Amount_LC`, `Invoiced_Value_LC`, `Invoiced_Quantity` | `Fact_PO_Header` |
| 2 | `First_GR_Date` (from `Fact_OTD`), `Related_Invoice_Receipt_Items`, `Invoice_Status` (roll-up); `First_GR_Date`, `Invoice_Receipt_Item`, `Invoice_Status`; `Received_Quantity` | `Dim_PO_Header`; `Dim_PO_Item`; `Fact_PO_Header` |
| 3 | Item: `Item_Value(_LC)`, `Price_Per_Unit_DC/_LC`, `Material_Plant_Price_Per_Unit` (S → standard, else moving), `Purchase_Value`, `Material_Value`, `GR_Amount_LC`, `Received_Value_DC`, `Received/Invoiced_Quantity`, `Ordered/Received/Invoiced_Quantity_Base_UOM`, `Quantity_Open_Active`. Header: the sums of these, `Net_Order_Value(_USD_FIX/_EOM)`, `Quantity_Open_Active_Base_UOM` | `Fact_PO_Delivery_Status` (+ `Fact_PO_LineItem`, old columns only); `Fact_PO_Header` |
| 4 | `Goods_Receipt_Date`, `Received_Quantity`, `Quantity_Base_UOM`, `Order_Due_Date`, `Planned_Date` | `Fact_PO_LineItem_Schedule` |

**Removed 18 Nov 2026** (business: not needed, or replaced): `Expected_Delivery_Date`, `MRP_Relevant`, `Planned_Goods_Avail_Date` (not needed); `Delivery_Reliability_Pct` on the schedule fact and the header (use `Fact_OTD` in SM_AUDIT); `Goods_Receipt_Status` (use `Order_Status_Purchase`); the schedule fact's `Received_Value_LC` / `Invoiced_Quantity` (item values repeated per line, no report uses them), `Quantity_Closed`, `Received_Quantity_Base_UOM`, `Invoiced_Quantity_Base_UOM` (the approved quantities are at item and PO level).

Prerequisites: PROC Silver EKPO run once with the base-unit factor (`NB_Config_PROC`, EKPO `enrich`: `Base_Factor_PO`, `Base_Factor_MARM`); without it the `*_Base_UOM` columns stay empty and a warning is printed. EKES `KZDIS` live for `Order_Due_Date`.

In [ ]:
# ── Received and invoiced value on the PO header (reports 09 and 14), from the PO history ─────────────────────
# DECISIONS (made, reasons in brackets):
#   D1 Home: Fact_PO_Header for the PO-grain reports (09, 14 "PD Header"); item values on Fact_PO_Delivery_Status (cell 3).
#   D2 Received_Value_DC = goods receipt quantity (type 1) x PO net price per unit (EKPO Price_Per_Unit), document currency.
#      [Approved 18 Nov 2026: report 09 "Received Value", = GR Value in ZMMPO0013. Renamed from Received_Value_LC: EKPO
#      Price_Per_Unit is NETPR / PEINH, so the value is in the PO's document currency, not local currency.]
#   D3 Invoiced_Value_LC = invoice receipt amount in local currency (EKBE type 2, Signed_Amount_LC, net of reversals).
#   D3b GR_Amount_LC = goods receipt amount in local currency (EKBE type 1, Signed_Amount_LC, net of reversals).
#      [Approved 18 Nov 2026: report 14 "PD Header / Received Value" and "PD Item / Received Value" (cell 3).]
#   D4 Transaction types: 1 = goods receipt, 2 = invoice receipt. [Same as Fact_OTD and Fact_PO_Receipts.]
#   D5 Free items (price 0) give zero. Stock-transfer POs (supplying plant on the header) give no Received_Value_DC (empty).
#   D6 This block re-writes Fact_PO_Header (overwrite). Folding it into the build cell is a separate task.
# Re-read from Gold (1 Nov 2026): the DataFrame left by the build cell still points at the table as it was before the earlier overwrite
# (stable_sk / gold() reads in its plan); writing from it after columns were added fails with DELTA_SCHEMA_CHANGE_SINCE_ANALYSIS.
fact_po_hdr = gold('Fact_PO_Header')
_rc      = gold('Fact_PO_Receipts').select('PO_SK', 'PO_Item_SK', 'Transevent_Type', 'Signed_Quantity', 'Signed_Amount_LC')
_price   = silver('EKPO').select('PO_Item_SK', F.col('Price_Per_Unit').cast('double').alias('_price'))
_supply  = silver('EKKO').select('PO_SK', F.trim(F.coalesce(F.col('Supplying_Plant'), F.lit(''))).alias('_supply_plant'))
_rc_item = (_rc.join(_price, 'PO_Item_SK', 'left').join(_supply, 'PO_SK', 'left')
    .withColumns({
        '_recv_val': F.when((F.col('Transevent_Type') == '1') & (F.col('_supply_plant') == ''),                 # D5
                            F.coalesce(F.col('Signed_Quantity') * F.col('_price'), F.lit(0.0))),              # D2
        '_gr_amt':   F.when(F.col('Transevent_Type') == '1', F.col('Signed_Amount_LC')),                            # D3b
        '_inv_val':  F.when(F.col('Transevent_Type') == '2', F.col('Signed_Amount_LC')),                            # D3
        '_inv_qty':  F.when(F.col('Transevent_Type') == '2', F.col('Signed_Quantity')),
    }))
_po_val  = (_rc_item.groupBy('PO_SK')
    .agg(F.sum('_recv_val').alias('Received_Value_DC'),                                                              # D2
         F.sum('_gr_amt').alias('GR_Amount_LC'),                                                                     # D3b
         F.sum('_inv_val').alias('Invoiced_Value_LC'),                                                               # D3
         F.sum('_inv_qty').alias('Invoiced_Quantity')))
# Re-runnable: the columns this cell adds are dropped first (Received_Value_LC too: renamed 18 Nov 2026).
fact_po_hdr = (fact_po_hdr.drop('Received_Value_LC', 'Received_Value_DC', 'GR_Amount_LC', 'Invoiced_Value_LC', 'Invoiced_Quantity')
               .join(_po_val, 'PO_SK', 'left'))
n = write_gold(fact_po_hdr, 'Fact_PO_Header', 'PO_SK', mode='overwrite')   # D6

In [ ]:
# ── Receipt and invoice fields on the PO dimensions and header fact (reports 09, 11, 15) ─────────────────────
# Fields: First_GR_Date, Related_Invoice_Receipt_Items, Invoice_Status (Dim_PO_Header); First_GR_Date, Invoice_Receipt_Item,
# Invoice_Status (Dim_PO_Item); Received_Quantity (Fact_PO_Header). Must run after the Fact_OTD cell: it reads Fact_OTD.
# DECISIONS (made, reasons in brackets):
#   D13 Dates, texts and statuses go on the dimensions, the quantity on the fact. [Audit design: numbers in facts only.]
#   D14 First_GR_Date = earliest receipt date in Fact_OTD, per PO (Dim_PO_Header) and per item (Dim_PO_Item).
#       [18 Nov 2026, business: "should be available from OTD data … probably needs calculated as an item level dimension".
#       Fact_OTD nets reversals, so a reversed receipt does not set the date.]
#   D15 Related_Invoice_Receipt_Items / Invoice_Receipt_Item = invoice document / year / item (type 2), item zeros stripped, sorted,
#       joined with ', '. [Approved 18 Nov 2026: "concat if multiple invoices? yes". Report 11 reads the item's list through PO_Item_SK.]
#   D16 Received_Quantity = sum of Signed_Quantity of type 1 (receipts net of reversals), in order units.
#   D17 Invoice_Status per item (approved 18 Nov 2026, reports 09 and 15):
#         no invoice receipt expected (EKPO REPOS blank)                                   → Not applicable
#         no invoiced quantity                                                              → Open
#         |invoiced qty| >= |ordered qty| (|received qty| when GR-based invoice verification,
#         EKPO WEBRE = X), or final invoice set (EKPO EREKZ = X)                            → Closed
#         otherwise                                                                         → Partially open
#       PO header = roll-up of the item statuses, items with deletion flag L left out (as Execution_Status on Fact_PO_Header):
#       all Not applicable → Not applicable; all Closed → Closed; all Open → Open; otherwise Partially open.
#       [Business approved the header "for now"; summing quantities across items would mix units, so the items are rolled up.]
#   D6  Re-writes Dim_PO_Header, Dim_PO_Item and Fact_PO_Header (overwrite).
dim_po_hdr  = gold('Dim_PO_Header')
dim_po_item = gold('Dim_PO_Item')
fact_po_hdr = gold('Fact_PO_Header')
_gr = gold('Fact_PO_Receipts').select('PO_SK', 'PO_Item_SK', 'Transevent_Type', 'DebitCredit_Ind', 'Posting_Date', 'Signed_Quantity',
                                      'Material_Document', 'Material_Doc_Year', 'Material_DocItem')
_otd = gold('Fact_OTD').select('PO_SK', 'PO_Item_SK', 'Receipt_Date')
_po_first = _otd.groupBy('PO_SK').agg(F.min('Receipt_Date').alias('First_GR_Date'))                                            # D14
_it_first = _otd.groupBy('PO_Item_SK').agg(F.min('Receipt_Date').alias('First_GR_Date'))                                       # D14
_ir_key = F.concat_ws('/', 'Material_Document', 'Material_Doc_Year', F.regexp_replace(F.col('Material_DocItem'), r'^0+', ''))   # D15
_po_gr  = (_gr.filter(F.col('Transevent_Type') == '1').groupBy('PO_SK')
    .agg(F.sum('Signed_Quantity').alias('Received_Quantity')))                                                                  # D16
_po_ir  = (_gr.filter(F.col('Transevent_Type') == '2').groupBy('PO_SK')
    .agg(F.array_join(F.array_sort(F.collect_set(_ir_key)), ', ').alias('Related_Invoice_Receipt_Items')))                      # D15
_it_ir  = (_gr.filter(F.col('Transevent_Type') == '2').groupBy('PO_Item_SK')
    .agg(F.array_join(F.array_sort(F.collect_set(_ir_key)), ', ').alias('Invoice_Receipt_Item')))                               # D15
# D17 Invoice_Status
_it_q = (_gr.groupBy('PO_Item_SK')
    .agg(F.sum(F.when(F.col('Transevent_Type') == '2', F.col('Signed_Quantity'))).alias('_ir_qty'),
         F.sum(F.when(F.col('Transevent_Type') == '1', F.col('Signed_Quantity'))).alias('_gr_qty')))
_flag = lambda c: F.upper(F.trim(F.coalesce(F.col(c), F.lit(''))))
_cmp  = F.when(_flag('GR_Based_Inv_Verif') == 'X', F.coalesce(F.col('_gr_qty'), F.lit(0.0))).otherwise(F.coalesce(F.col('Order_Quantity'), F.lit(0.0)))
_it_iv = (silver('EKPO').select('PO_Item_SK', 'Order_Quantity', 'Invoice_Receipt_Ind', 'GR_Based_Inv_Verif', 'Final_Invoice', 'Deletion_Indicator')
    .join(_it_q, 'PO_Item_SK', 'left')
    .withColumn('Invoice_Status',
        F.when(_flag('Invoice_Receipt_Ind') == '', 'Not applicable')
         .when(F.coalesce(F.col('_ir_qty'), F.lit(0.0)) == 0, 'Open')
         .when((F.abs(F.col('_ir_qty')) >= F.abs(_cmp)) | (_flag('Final_Invoice') == 'X'), 'Closed')
         .otherwise('Partially open')))
_live = _it_iv.join(gold('Fact_PO_Delivery_Status').select('PO_SK', 'PO_Item_SK'), 'PO_Item_SK').filter(_flag('Deletion_Indicator') != 'L')
_po_iv = (_live.groupBy('PO_SK')
    .agg(F.sum((F.col('Invoice_Status') != 'Not applicable').cast('int')).alias('_n'),
         F.sum((F.col('Invoice_Status') == 'Closed').cast('int')).alias('_c'),
         F.sum((F.col('Invoice_Status') == 'Open').cast('int')).alias('_o'))
    .select('PO_SK', F.when(F.col('_n') == 0, 'Not applicable').when(F.col('_c') == F.col('_n'), 'Closed')
                      .when(F.col('_o') == F.col('_n'), 'Open').otherwise('Partially open').alias('Invoice_Status')))
# Re-runnable: the columns this cell adds are dropped first, so a second run of the cell (or a rerun after an error) does not duplicate them.
dim_po_hdr  = (dim_po_hdr.drop('First_GR_Date', 'Related_Invoice_Receipt_Items', 'Invoice_Status')
               .join(_po_first, 'PO_SK', 'left').join(_po_ir, 'PO_SK', 'left').join(_po_iv, 'PO_SK', 'left'))
dim_po_item = (dim_po_item.drop('Invoice_Receipt_Item', 'First_GR_Date', 'Invoice_Status')
               .join(_it_first, 'PO_Item_SK', 'left').join(_it_ir, 'PO_Item_SK', 'left')
               .join(_it_iv.select('PO_Item_SK', 'Invoice_Status'), 'PO_Item_SK', 'left'))
fact_po_hdr = fact_po_hdr.drop('Received_Quantity').join(_po_gr, 'PO_SK', 'left')
n = write_gold(dim_po_hdr,  'Dim_PO_Header',  'PO_SK',      mode='overwrite')   # D6
n = write_gold(dim_po_item, 'Dim_PO_Item',    'PO_Item_SK', mode='overwrite')   # D6
n = write_gold(fact_po_hdr, 'Fact_PO_Header', 'PO_SK',      mode='overwrite')   # D6

In [ ]:
# ── Item values, prices and quantities (reports 09, 11, 14), and their totals on the PO header ─────────────────────
# Item (Fact_PO_Delivery_Status): Item_Value, Item_Value_LC, Price_Per_Unit_DC, Price_Per_Unit_LC, Material_Plant_Price_Per_Unit,
# Purchase_Value, Material_Value, GR_Amount_LC, Received_Value_DC, Received_Quantity, Invoiced_Quantity, Ordered_Quantity_Base_UOM,
# Received_Quantity_Base_UOM, Invoiced_Quantity_Base_UOM, Quantity_Open_Active. PO header (Fact_PO_Header): their sums.
# Needs _rc_item from cell 1 in the same session.
# DECISIONS (made, reasons in brackets):
#   D18 Fact_PO_LineItem (the duplicate slated for removal) keeps getting only the columns it had (plus the rename and the price fix);
#       the 18 Nov 2026 columns go on Fact_PO_Delivery_Status only. [No report reads Fact_PO_LineItem any more.]
#   D19 Item_Value = Net_Order_Value (EKPO NETWR, document currency). [The approved text (Order_Quantity x factor x Price_Per_Unit_USD_FIX)
#       multiplies a base-unit quantity by a price per order unit; kept as NETWR and asked back to the business.]
#       Price_Per_Unit_DC = Net_Order_Value / (Order_Quantity x base factor): price per BASE unit, document currency
#       [approved 18 Nov 2026, report 14 "PD Item / Price per Unit in DC"; was EKPO NETPR / PEINH, per order price unit].
#   D20 Local currency = document currency x the EKKO exchange rate (WKURS): Item_Value_LC, Price_Per_Unit_LC (per base unit, approved
#       18 Nov 2026). A rate of 0 or empty gives an empty value, not zero.
#   D21 Material_Plant_Price_Per_Unit = MBEW standard price per unit when price control is S, else moving average price per unit
#       (MD Silver MBEW, price / PEINH). [Approved 18 Nov 2026, "Material on Plant Level / Price per Unit"; was moving price only.]
#   D22 Received_Quantity / Invoiced_Quantity = EKBE types 1 / 2 per item, net of reversals, order units. Received_Value_DC = receipt
#       quantity x PO net price (header rule D2, renamed from Received_Value_LC). GR_Amount_LC = EKBE type 1 amount (D3b).
#   D32 Base unit (approved 18 Nov 2026): quantity x COALESCE(Base_Factor_PO, Base_Factor_MARM), both from PROC Silver EKPO
#       (UMREZ / UMREN, else MARM of the order unit). No factor → empty, counted in a warn-only check; never the raw quantity.
#       Ordered = Order_Quantity; Received (= report 14 "Quantity Closed") = Delivered (Σ EKET WEMNG); Invoiced = Invoiced_Quantity.
#   D33 Purchase_Value = Order_Quantity x Price_Per_Unit over every item, deleted or cancelled included (approved text);
#       on the item and summed on the header [Jeremy: "maybe add it to both"].
#   D34 Material_Value = Ordered_Quantity_Base_UOM x Material_Plant_Price_Per_Unit; header = sum over items.
#   D35 Quantity_Open_Active = 0 when the item is Closed or Cancelled_*, else Quantity_Open (order unit). A new column: Quantity_Open
#       keeps its meaning for SM_PROC. Header Quantity_Open_Active_Base_UOM = sum of the item value x base factor, in base unit like
#       the other header quantities. [The approved header text sums order units across items; asked back to the business.]
#   D36 Header: Net_Order_Value (+ _USD_FIX / _USD_EOM) = sum over every item [report 09 "Ordered Value", report 14 "PD Header /
#       Ordered Value"]; Ordered_Quantity_Base_UOM leaves out items with deletion flag L (approved text); the others sum every item.
#   D6  Re-writes Fact_PO_Delivery_Status, Fact_PO_LineItem and Fact_PO_Header (overwrite).
base = gold('Fact_PO_Delivery_Status')
_q_item = (_rc_item.groupBy('PO_Item_SK')
    .agg(F.sum(F.when(F.col('Transevent_Type') == '1', F.col('Signed_Quantity'))).alias('Received_Quantity'),                  # D22
         F.sum(F.when(F.col('Transevent_Type') == '2', F.col('Signed_Quantity'))).alias('Invoiced_Quantity'),                   # D22
         F.sum('_recv_val').alias('Received_Value_DC'),                                                                         # D22
         F.sum('_gr_amt').alias('GR_Amount_LC')))                                                                               # D22
_mbew_pc = (read_table('md_silver', 'MBEW', active=True)                                                                         # D21
    .select(_NZ(F.col('Material')).alias('_pc_mat'), _NZ(F.col('Plant')).alias('_pc_plant'),
            F.when(F.upper(F.trim(F.col('Price_Control'))) == 'S', F.col('Standard_Price_Per_Unit'))
             .otherwise(F.col('Moving_Avg_Price_Per_Unit')).cast('double').alias('Material_Plant_Price_Per_Unit'))
    .dropDuplicates(['_pc_mat', '_pc_plant']))
# ▶ 19 Nov 2026: the item facts carry Material_SK / Plant_SK only (drop_redundant removes Material and Plant), so the item's material
#   and plant for the price join come from PROC Silver EKPO
_item_mp = (silver('EKPO').select('Purchasing_Document', 'Item', _NZ(F.col('Material')).alias('_it_mat'), _NZ(F.col('Plant')).alias('_it_plant'))
            .dropDuplicates(['Purchasing_Document', 'Item']))
_rate = F.when(F.col('Exchange_Rate') > 0, F.col('Exchange_Rate'))                                                              # D20
_q_cols = ['Received_Quantity', 'Invoiced_Quantity', 'Received_Value_DC', 'GR_Amount_LC']
_old    = ['Received_Value_LC', 'Material_Plant_Price_Per_Unit', 'Item_Value', 'Item_Value_LC', 'Price_Per_Unit_DC', 'Price_Per_Unit_LC', '_bf']

def _base_factor(df):
    # D32 base-unit factor from PROC Silver EKPO (flows into the item facts with the other EKPO columns)
    cols = [c for c in ('Base_Factor_PO', 'Base_Factor_MARM') if c in df.columns]
    if not cols:
        print('  WARNING: Base_Factor_PO / Base_Factor_MARM not on the item fact: run PROC Silver EKPO (base-unit factor, 18 Nov 2026) '
              'first; the *_Base_UOM columns, Material_Value and the prices per base unit stay empty')
        return F.lit(None).cast('double')
    return F.coalesce(*[F.col(c) for c in cols]) if len(cols) > 1 else F.col(cols[0])

def _with_item_values(df):
    _base_qty = F.col('Order_Quantity') * F.col('_bf')
    return (df.drop(*[c for c in _q_cols + _old if c in df.columns]).join(_q_item, 'PO_Item_SK', 'left')
              .join(_item_mp, ['Purchasing_Document', 'Item'], 'left')
              .join(_mbew_pc, (F.col('_it_mat') == F.col('_pc_mat')) & (F.col('_it_plant') == F.col('_pc_plant')), 'left')
              .drop('_pc_mat', '_pc_plant', '_it_mat', '_it_plant')
              .withColumn('_bf', _base_factor(df))
              .withColumns({
                  'Item_Value':        F.col('Net_Order_Value'),                                                                     # D19
                  'Item_Value_LC':     F.col('Net_Order_Value') * _rate,                                                            # D20
                  'Price_Per_Unit_DC': F.when(_base_qty != 0, F.col('Net_Order_Value') / _base_qty),                                # D19
              })
              .withColumn('Price_Per_Unit_LC', F.col('Price_Per_Unit_DC') * _rate))                                                # D20

_new_cols = ['Purchase_Value', 'Material_Value', 'Ordered_Quantity_Base_UOM', 'Received_Quantity_Base_UOM', 'Invoiced_Quantity_Base_UOM',
             'Quantity_Open_Active']
base = (_with_item_values(base.drop(*[c for c in _new_cols if c in base.columns]))
    .withColumns({
        'Purchase_Value':             F.col('Order_Quantity') * F.col('Price_Per_Unit'),                                       # D33
        'Quantity_Open_Active':       F.when(F.col('Execution_Status').isin('Closed', 'Cancelled_Header', 'Cancelled_Line'), F.lit(0.0))
                                       .otherwise(F.col('Quantity_Open')),                                                      # D35
    })
    .withColumns({
        'Ordered_Quantity_Base_UOM':  F.col('Order_Quantity') * F.col('_bf'),                                                   # D32
        'Received_Quantity_Base_UOM': F.col('Delivered') * F.col('_bf'),                                                        # D32
        'Invoiced_Quantity_Base_UOM': F.col('Invoiced_Quantity') * F.col('_bf'),                                                # D32
    })
    .withColumn('Material_Value', F.col('Ordered_Quantity_Base_UOM') * F.col('Material_Plant_Price_Per_Unit')))                # D34
_no_factor = base.filter(F.col('Order_Quantity').isNotNull() & F.col('_bf').isNull()).count()
print(f'  Items without a base-unit factor (base quantities empty): {_no_factor:,}')
if _no_factor:
    dq_record('GOLD', 'Fact_PO_Delivery_Status', 'base_unit_factor', 'WARN',
              f'{_no_factor:,} items have no base-unit factor (EKPO UMREZ / UMREN and MARM): base-unit quantities and Material_Value empty')
# D36 PO totals
_usd = [c for c in ('Net_Order_Value_USD_FIX', 'Net_Order_Value_USD_EOM') if c in base.columns]
_hdr_tot = (base.groupBy('PO_SK')
    .agg(F.sum('Net_Order_Value').alias('Net_Order_Value'), *[F.sum(c).alias(c) for c in _usd],
         F.sum('Purchase_Value').alias('Purchase_Value'),
         F.sum('Material_Value').alias('Material_Value'),
         F.sum(F.when(F.upper(F.trim(F.coalesce(F.col('Deletion_Indicator'), F.lit('')))) != 'L', F.col('Ordered_Quantity_Base_UOM')))
          .alias('Ordered_Quantity_Base_UOM'),
         F.sum('Received_Quantity_Base_UOM').alias('Received_Quantity_Base_UOM'),
         F.sum('Invoiced_Quantity_Base_UOM').alias('Invoiced_Quantity_Base_UOM'),
         F.sum(F.col('Quantity_Open_Active') * F.col('_bf')).alias('Quantity_Open_Active_Base_UOM')))
base = base.drop('_bf')
n = write_gold(base, 'Fact_PO_Delivery_Status', 'PO_Item_SK', mode='overwrite')   # D6
# Fact_PO_LineItem is the duplicate slated for removal (25 Oct 2026): updated only while its cell still runs in this notebook (D18)
if table_exists('gold', 'Fact_PO_LineItem'):
    fact_li = _with_item_values(gold('Fact_PO_LineItem')).drop('_bf')
    n = write_gold(fact_li, 'Fact_PO_LineItem', 'PO_Item_SK', mode='overwrite')   # D6, D18
else:
    print('  Fact_PO_LineItem not built in this notebook (removed): item columns written to Fact_PO_Delivery_Status only')
# Header: Delivery_Reliability_Pct left the table on 18 Nov 2026 (reliability now comes from Fact_OTD in SM_AUDIT)
_hdr_cols = [c for c in _hdr_tot.columns if c != 'PO_SK']
fact_po_hdr = gold('Fact_PO_Header')
fact_po_hdr = (fact_po_hdr.drop(*[c for c in _hdr_cols + ['Delivery_Reliability_Pct'] if c in fact_po_hdr.columns])
               .join(_hdr_tot, 'PO_SK', 'left'))
n = write_gold(fact_po_hdr, 'Fact_PO_Header', 'PO_SK', mode='overwrite')   # D6

In [ ]:
# ── Schedule line fields (report 11): receipts, quantity in base unit, due and planned dates ───────────────────────────────
# Fields: Goods_Receipt_Date, Received_Quantity, Quantity_Base_UOM, Order_Due_Date, Planned_Date. Must run after the Fact_OTD cell:
# it reads Fact_OTD. Needs the EKES KZDIS config block (NB_Config_PROC) for Order_Due_Date.
# DECISIONS (made, reasons in brackets):
#   D23 Receipts per schedule line come from Fact_OTD, which already assigns receipts to lines and nets out reversals.
#   D24 Goods_Receipt_Date = latest receipt date assigned to the line [approved 18 Nov 2026: "max Receipt_Date"];
#       Received_Quantity = quantity assigned to the line.
#   D27 Quantity_Base_UOM = Scheduled_Quantity x COALESCE(Base_Factor_PO, Base_Factor_MARM) from PROC Silver EKPO (18 Nov 2026: was a
#       Gold-side factor with a default of 1). No factor → empty.
#   D28 Order_Due_Date = latest MRP-relevant confirmation date of the item (EKES, KZDIS = 'X'); else for stock transfers the
#       committed date when a committed quantity exists; else the scheduled delivery date. Empty on over-delivery lines.
#   D29 Planned_Date = Order_Due_Date.
#   Removed 18 Nov 2026 (business feedback): Expected_Delivery_Date (not needed), Goods_Receipt_Status (use Order_Status_Purchase),
#   Delivery_Reliability_Pct (use Fact_OTD in SM_AUDIT), Quantity_Closed, Received_Quantity_Base_UOM, Invoiced_Quantity_Base_UOM
#   and the former cell's Received_Value_LC / Invoiced_Quantity (item values; the approved values are at item and PO level).
#   D6  Re-writes Fact_PO_LineItem_Schedule (overwrite).
fact_sa = gold('Fact_PO_LineItem_Schedule')
_otd = (gold('Fact_OTD').groupBy('PO_Schedule_SK')
    .agg(F.max('Receipt_Date').alias('Goods_Receipt_Date'),                                                                     # D24
         F.sum('Allocated_Qty').alias('Received_Quantity')))                                                                    # D24
_ekpo = silver('EKPO')
_bf_cols = [c for c in ('Base_Factor_PO', 'Base_Factor_MARM') if c in _ekpo.columns]
if not _bf_cols:
    print('  WARNING: PROC Silver EKPO has no Base_Factor_PO / Base_Factor_MARM yet: Quantity_Base_UOM stays empty')
_ek  = _ekpo.select('PO_Item_SK', (F.coalesce(*[F.col(c) for c in _bf_cols]) if len(_bf_cols) > 1 else
                                   F.col(_bf_cols[0]) if _bf_cols else F.lit(None).cast('double')).alias('_base_factor'))   # D27
_sto = silver('EKKO').select('PO_SK', (F.trim(F.coalesce(F.col('Supplying_Plant'), F.lit(''))) != '').alias('_is_sto'))
_mrp = (silver('EKES').filter(F.col('MRP_Relevant') == 'X')
    .groupBy('PO_Item_SK').agg(F.max('Confirmed_Delivery_Date').alias('_mrp_conf_date')))                                       # D28
# Re-runnable: the columns this cell adds are dropped first, and so are the columns removed on 18 Nov 2026 (overwrite drops them).
_removed = ['Quantity_Closed', 'Received_Quantity_Base_UOM', 'Invoiced_Quantity_Base_UOM', 'Goods_Receipt_Status', 'Delivery_Reliability_Pct',
            'Expected_Delivery_Date', 'Received_Value_LC', 'Invoiced_Quantity', 'MRP_Relevant', 'Planned_Goods_Avail_Date']
fact_sa = (fact_sa.drop('Goods_Receipt_Date', 'Received_Quantity', 'Quantity_Base_UOM', 'Order_Due_Date', 'Planned_Date',
                        *[c for c in _removed if c in fact_sa.columns])
                  .join(_otd, 'PO_Schedule_SK', 'left').join(_ek, 'PO_Item_SK', 'left')
                  .join(_sto, 'PO_SK', 'left').join(_mrp, 'PO_Item_SK', 'left'))
_sched = F.coalesce(F.col('Scheduled_Quantity'), F.lit(0.0))
_over  = (_sched == 0) & (F.coalesce(F.col('Quantity_Delivered'), F.lit(0.0)) > 0)
fact_sa = fact_sa.withColumns({
    'Quantity_Base_UOM': F.col('Scheduled_Quantity') * F.col('_base_factor'),                                                   # D27
    'Order_Due_Date':    F.when(~_over,
                             F.coalesce(F.col('_mrp_conf_date'),
                                        F.when(F.coalesce(F.col('_is_sto'), F.lit(False)) & (F.col('Committed_Quantity') > 0), F.col('Committed_Date')),
                                        F.col('Delivery_Date'))),                                                               # D28
})
fact_sa = fact_sa.withColumn('Planned_Date', F.col('Order_Due_Date')).drop('_base_factor', '_is_sto', '_mrp_conf_date')        # D29
n = write_gold(fact_sa, 'Fact_PO_LineItem_Schedule', 'PO_Schedule_Status_SK', mode='overwrite')   # D6

In [ ]:
# ── New facts from EKPV / EBKN / EBAN ─────────────────────────────────────────

print("\nBuilding Fact_PO_Stock_Transfer_Shipping (EKPV)...")

ekpv = silver("EKPV")

# Column ordering: SKs → business keys → rest → metadata
sk_cols   = [c for c in ("PO_Stock_SK",) if c in ekpv.columns]
# Purchasing_Document_Item: was Purchasing_Docment_Item, renamed in the config on 6 Nov 2026
pk_cols   = [c for c in ("Purchasing_Document", "Purchasing_Document_Item") if c in ekpv.columns]
rest_cols = [c for c in ekpv.columns if c not in sk_cols + pk_cols and not c.startswith("_")]
meta_cols = [c for c in ekpv.columns if c.startswith("_")]

fact_stk = ekpv.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_stk, "Fact_PO_Stock_Transfer_Shipping", "PO_Stock_SK")
print(f"  Fact_PO_Stock_Transfer_Shipping rows built: {n}")


print("\nBuilding Fact_Purch_Req_Account_Assignment (EBKN)...")

ebkn = silver("EBKN")

sk_cols   = [c for c in ("Purchase_Requisition_Account_SK",) if c in ebkn.columns]
pk_cols   = [c for c in ("Purchase_Requisition", "Item_Of_Requisition", "Serial_Number_Account_Assignment") if c in ebkn.columns]
rest_cols = [c for c in ebkn.columns if c not in sk_cols + pk_cols and not c.startswith("_")]
meta_cols = [c for c in ebkn.columns if c.startswith("_")]

fact_pr_acct = ebkn.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_pr_acct, "Fact_Purch_Req_Account_Assignment", "Purchase_Requisition_Account_SK")
print(f"  Fact_Purch_Req_Account_Assignment rows built: {n}")


print("\nBuilding Fact_Purch_Req (EBAN)...")

eban = silver("EBAN")

# EBAN Silver already carries SKs from MD (Material_SK, Material_Plant_SK, ...)
sk_cols   = [c for c in ("Purchase_Requisition_SK", "Material_SK", "Material_Plant_SK") if c in eban.columns]
pk_cols   = [c for c in ("Purchase_Requisition", "Item_Of_Requisition") if c in eban.columns]
rest_cols = [c for c in eban.columns if c not in sk_cols + pk_cols and not c.startswith("_")]
meta_cols = [c for c in eban.columns if c.startswith("_")]

fact_pr = eban.select(sk_cols + pk_cols + rest_cols + meta_cols)

n = write_gold(fact_pr, "Fact_Purch_Req", "Purchase_Requisition_SK")
print(f"  Fact_Purch_Req rows built: {n}")

In [ ]:
# ── Gold checks: rows and amounts vs Silver, stale rows, referential integrity ─
gold_issues = run_gold_dq(GOLD_DQ, PROC_TABLE_CONFIGS)

In [ ]:
# ── End-of-run report: result, issues in plain language, tables, checks ────────
end_of_run_report()